# Configuração do Servidor Ollama e Modelos Locais

Esta seção inicializa o servidor local Ollama e realiza o download dos modelos necessários para embeddings (`nomic-embed-text`), geração de respostas (`qwen2.5:3b`) e avaliação automática (`qwen2.5:7b` para LLM-as-a-Judge).

In [46]:
# Installing ollama
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2

>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [47]:
# Running ollama server
import subprocess
import time
import requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up():
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)

print("Server running:", server_is_up())

Server running: True


In [48]:
# Pull models
# !ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:7b                 845dbda0ea48    4.7 GB    1 second ago              


# Dependências, Configuração de Caminhos e Provisionamento do Corpus

Instalação dos pacotes necessários (`chromadb`, `openai`, `tiktoken`, `langchain-text-splitters`), configuração de caminhos persistentes (Google Drive ou local) e provisionamento resiliente dos 10 manuais técnicos da Huawei Ascend.

In [49]:
# Installing dependencies
!pip -q install chromadb openai tiktoken pandas pyarrow tqdm requests langchain-text-splitters

In [50]:
import os, re, glob, json, math, time, shutil, urllib.request
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
import chromadb

# ============================================================================
# Google Drive & Local Directory Configuration
# ============================================================================
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/RAG_Atlas200DK")
    except Exception:
        BASE_DIR = Path(".")
else:
    BASE_DIR = Path(".")

PROCESSED_MD_DIR = BASE_DIR / "data" / "processed" / "markdown"
OUTPUT_DIR       = BASE_DIR / "data" / "output"
CHROMA_DIR       = str(BASE_DIR / "chroma_db")

for d in [PROCESSED_MD_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL   = "qwen2.5:7b"
JUDGE_MODEL = "qwen2.5:7b"
EMB_MODEL   = "nomic-embed-text"
CHUNK_STRATEGY = "atomic"
COLLECTION_NAME = f"huawei_ascend_{CHUNK_STRATEGY}_v3_col"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ============================================================================
# Resilient Corpus Asset Provisioning (10 Markdown Manuals + Chunk Dataset)
# ============================================================================
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ramonvfs/RAG_Atlas200DK/master/"
CHUNK_DATASET_STEM = "huawei_ascend_rag_chunks_atomic"

EXPECTED_DOCS = [
    "ApplicationDevelopmentGuide_MindStudio_.md",
    "ATCToolInstructions.md",
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_.md",
    "EnvironmentDeploymentGuide.md",
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
    "MindStudioInstallationGuideUbuntuX86.md",
    "TBECustomOperatorDevelopmentGuide_CLI_.md",
    "TensorFlowNetworkModelPortingandTrainingGuide.md",
    "UserGuide2020.md",
    "UserGuide2021.md",
]

ALT_ROOTS = [
    Path("."),
    Path("/content/RAG_Atlas200DK"),
    Path("/content"),
    Path("/content/drive/MyDrive/RAG_Atlas200DK"),
]

def _download_file(rel_path: str, dest: Path) -> bool:
    url = GITHUB_RAW_BASE + urllib.request.quote(rel_path)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as out_f:
            out_f.write(resp.read())
        print(f"  [DOWNLOADED] {rel_path}")
        return True
    except Exception as e:
        print(f"  [WARNING] Could not download {rel_path}: {e}")
        return False

def ensure_corpus_assets(md_dir: Path, out_dir: Path) -> tuple:
    # 1. Ensure Markdown manuals exist
    for name in EXPECTED_DOCS:
        target = md_dir / name
        if target.exists():
            continue
        for root in ALT_ROOTS:
            cand = root / "data" / "processed" / "markdown" / name
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {name} from {cand.parent}")
                break
        if not target.exists():
            _download_file(f"data/processed/markdown/{name}", target)

    md_paths = sorted(p for p in md_dir.glob("*.md") if p.is_file())

    # 2. Check for pre-computed chunk dataset (parquet preferred, then json)
    dataset_path = None
    for ext in [".parquet", ".json"]:
        target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
        if target.exists():
            dataset_path = target
            break
        for root in ALT_ROOTS:
            cand = root / "data" / "output" / f"{CHUNK_DATASET_STEM}{ext}"
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {target.name} from {cand.parent}")
                dataset_path = target
                break
        if dataset_path:
            break

    if dataset_path is None:
        for ext in [".parquet", ".json"]:
            target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
            if _download_file(f"data/output/{CHUNK_DATASET_STEM}{ext}", target):
                dataset_path = target
                break

    return md_paths, dataset_path

md_paths, chunk_dataset_path = ensure_corpus_assets(PROCESSED_MD_DIR, OUTPUT_DIR)
print(f"[OK] Located {len(md_paths)} Markdown manuals in {PROCESSED_MD_DIR}")
if chunk_dataset_path:
    print(f"[OK] Found pre-computed chunk dataset: {chunk_dataset_path.name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Located 10 Markdown manuals in /content/drive/MyDrive/RAG_Atlas200DK/data/processed/markdown
[OK] Found pre-computed chunk dataset: huawei_ascend_rag_chunks_atomic.parquet


# Estratégias de Chunking: Particionamento Atômico Ciente de Tabelas e Breadcrumbs (Estratégia 3 - Ativa)

Nesta célula, configuramos a **Estratégia 3 (Atomic Table-Aware)** como o particionador ativo do corpus:
- Rebaixa caixas de aviso (`# **NO TE**`, `#### NO TICE`) e pseudocabeçalhos de tabelas/figuras para texto inline em negrito antes do particionamento de seções, preservando a hierarquia estrutural real.
- Mantém tabelas de especificação e blocos de código atômicos e íntegros em chunks individuais.
- Anexa automaticamente até 3 linhas de prosa imediatamente anteriores (título/legenda da tabela) ao corpo da tabela, garantindo o contexto conceitual.
- Injeta breadcrumbs hierárquicos completos de contexto no formato `[Doc: <doc> > <caminho>]`.
- Subdivide seções de prosa que ultrapassam `max_chunk_tokens` através do `RecursiveCharacterTextSplitter`.
- Detecta e preserva marcadores estruturais (`is_table`, `is_code_block`) nos metadados de cada chunk.

A **Estratégia 2 (Markdown Header Splitting)** e a **Estratégia 1 (Recursive Window)** são mantidas implementadas, porém comentadas para referência e benchmarks comparativos.

In [51]:
import tiktoken
from typing import Optional
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Tokenizer wrapper using OpenAI cl100k_base
_ENC = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(_ENC.encode(text))

# ============================================================================
# Cell B.1 — Structural & Hierarchy Post-Processing Pipeline for Huawei Manuals
# ============================================================================

_HTML_TAG_RE = re.compile(r"</?(?:span|sup|sub|br|b|i|u|p|div)\b[^>]*>", re.I)
_MD_ESCAPE_RE = re.compile(r"\\([_*$#\[\](){}<>`|~^.!+\-])")

# Huawei proprietary footers, headers, and image placeholders to strip
_HEADER_FOOTER_PATTERNS = [
    re.compile(r"(?im)^.*Huawei\s+Proprietary.*$"),
    re.compile(r"(?im)^.*Huawei\s+Confidential.*$"),
    re.compile(r"(?im)^.*Huawei\s+Technologies\s+Co\.?,?\s*Ltd\.?.*$"),
    re.compile(r"(?im)^\s*Issue\s+\d+.*Date.*$"),
    re.compile(r"(?im)^\s*Copyright\s+©.*Huawei.*$"),
    re.compile(r"(?im)^\s*Page\s+\d+\s*$"),
    re.compile(r"(?im)^\s*\d+\s*$"),
    re.compile(r"(?im)^\s*<!--\s*image\s*-->\s*$"),
]

# Technical terms commonly broken by PDF line wraps
_HYPHEN_REJOINS = [
    (re.compile(r"DaV-\s*inci", re.I), "DaVinci"),
    (re.compile(r"NC1HW-\s*C0", re.I), "NC1HWC0"),
    (re.compile(r"conv2-\s*d", re.I), "conv2d"),
    (re.compile(r"Ascend-\s*CL", re.I), "AscendCL"),
    (re.compile(r"Mind-\s*Studio", re.I), "MindStudio"),
    (re.compile(r"Tensor-\s*Flow", re.I), "TensorFlow"),
    (re.compile(r"FRACTAL-\s*Z", re.I), "FRACTAL_Z"),
    (re.compile(r"LPDDR-?\s*4X", re.I), "LPDDR4X"),
    (re.compile(r"(\w)-\n(\w)"), r"\1\2"),
]

# Unit normalization: ensure consistent spacing/casing
_UNIT_NORMALIZATIONS = [
    (re.compile(r"(?<!\w)(T\s*F\s*L\s*O\s*P\s*S)(?!\w)", re.I), "TFLOPS"),
    (re.compile(r"(?<!\w)(T\s*O\s*P\s*S)(?!\w)", re.I), "TOPS"),
    (re.compile(r"(\d)\s*(GB\s*/\s*s)", re.I), r"\1 GB/s"),
    (re.compile(r"(\d)\s*(mA)(?!\w)", re.I), r"\1 mA"),
    (re.compile(r"(\d)\s*([VW])(?!\w)"), r"\1 \2"),
]

# Canonical chapter/section titles for Huawei manuals when stylized chapter banners are missed
_FALLBACK_SECTION_TITLES = {
    "ATCToolInstructions": {
        "1": "1 Introduction",
        "2": "2 Restrictions and Parameters",
        "3": "3 How to Use",
        "4": "4 AIPP Configuration",
        "5": "5 Single-Operator JSON File Configuration",
        "6": "6 Supported Operators",
        "7": "7 Special Topics",
        "8": "8 FAQs",
        "9": "9 Appendix",
    },
    "ApplicationDevelopmentGuide_MindStudio_": {
        "1": "1 Introduction",
        "2": "2 Getting Started",
        "3": "3 Application Development",
        "3.3": "3.3 Implementing a Project",
        "4": "4 FAQs",
        "5": "5 Appendix",
    },
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_": {
        "1": "1 Overview",
        "2": "2 Process Orchestration Overview",
        "3": "3 DDK Environment Setup",
        "4": "4 Offline Model Conversion",
        "5": "5 Operator Development",
        "6": "6 Application Software Development",
        "7": "7 Auxiliary Tool",
        "8": "8 API Reference",
        "9": "9 Command Reference",
        "10": "10 Appendix",
    },
    "EnvironmentDeploymentGuide": {
        "1": "1 Introduction",
        "2": "2 Preparing Accessories and a Development Server",
        "3": "3 Installing Hardware",
        "4": "4 Obtaining Software Packages",
        "5": "5 Setting Up the Hardware Environment",
        "6": "6 Setting up the Development Environment",
        "7": "7 Verifying the Environment",
        "8": "8 Service Memory Control with Cgroup",
        "9": "9 Common Operations",
        "10": "10 FAQs",
        "11": "11 Appendix",
    },
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08": {
        "1": "1 Product Introduction",
        "2": "2 Product Features",
        "3": "3 Product Specifications",
        "4": "4 Ports and Indicator",
        "5": "5 Certifications",
        "6": "6 Warranty",
        "A": "A Appendix",
    },
    "MindStudioInstallationGuideUbuntuX86": {
        "1": "1 Introduction",
        "2": "2 Privacy Statement",
        "3": "3 Obtaining Software Packages",
        "4": "4 Environment Preparation",
        "5": "5 MindStudio Installation",
        "6": "6 Upgrade",
        "7": "7 Uninstallation",
        "8": "8 FAQs",
        "9": "9 Appendix",
    },
    "TBECustomOperatorDevelopmentGuide_CLI_": {
        "1": "1 Introduction",
        "2": "2 Intended Audience",
        "3": "3 Precautions for Operator Development",
        "4": "4 Environment Preparation",
        "5": "5 Development Workflow",
        "6": "6 Creating a Custom Operator Development Project",
        "7": "7 Developing a Custom Operator",
        "8": "8 Developing the Plug-In of a Custom Operator",
        "9": "9 Loading a Plug-In for Model Conversion",
        "10": "10 Developing an Application",
        "11": "11 Common Operations",
        "12": "12 Appendix",
    },
    "TensorFlowNetworkModelPortingandTrainingGuide": {
        "1": "1 Introduction",
        "2": "2 Restrictions and Limitations",
        "3": "3 Migration Workflow",
        "4": "4 Auto Migration and Manual Migration",
        "5": "5 Model Training",
        "6": "6 Performance Tuning",
        "7": "7 Accuracy Tuning",
        "8": "8 Single-Server Multi-Device Training",
        "9": "9 More Functions",
        "10": "10 Reference",
        "11": "11 FAQs",
        "12": "12 Appendixes",
    },
    "UserGuide2020": {
        "1": "1 Introduction",
        "2": "2 Preparing Accessories and a Development Server",
        "3": "3 Setting Up the Hardware Environment",
        "4": "4 Setting Up the Development Environment",
        "5": "5 Configuring and Managing the Atlas 200 DK",
        "6": "6 Running Sample Applications",
        "7": "7 Common Operations",
        "8": "8 FAQs",
    },
    "UserGuide2021": {
        "1": "1 Introduction",
        "2": "2 Preparing Accessories and a Development Server",
        "3": "3 Hardware Installation",
        "4": "4 Software Installation",
        "5": "5 Common Operations",
        "6": "6 FAQs",
    },
}

_SUBSECTION_LABELS_RE = re.compile(
    r"^\s*(?:Hardware\s+Preparation|Software\s+Preparation|Environment\s+Preparation"
    r"|Preparing\s+Accessories|Preparing\s+a\s+Server|Prerequisites?"
    r"|Procedure|Context|Precautions(?:\s+for\s+[\w\s\-]+)?|Note|Notice|Caution|Warning"
    r"|Symptom|Possible\s+Causes?|Cause(?:\s+Analysis)?|Root\s+Cause|Fault\s+Locating"
    r"|Handling\s+Process|Impact\s+on\s+the\s+System|Solutions?"
    r"|Parameters?(?:\s+Description)?|Return\s+Values?|Calling\s+Examples?"
    r"|Format|Description|Example|Examples|Sample\s+Code"
    r"|Prototype|Function|Restrictions?|Rules?|API\s+Calling\s+Process"
    r"|Related\s+Operations|Follow-up\s+Operations|Follow-up\s+Procedure"
    r"|Verification|Reference|Others|Step\s+\d+\b.*)\s*$",
    re.I,
)

# Matches config/code comment lines starting with '# ' (e.g. in AIPP templates)
_CONFIG_COMMENT_HEADER_RE = re.compile(
    r"^#{1,6}\s*\**\s*(?:[a-z_][a-z0-9_]*\b.*|={3,}.*|\|\s*[A-Za-z0-9_]+\s*\|.*|Value\s*:.*|Type\s*:.*|Value\s+range\s*:.*|"
    r"If\s+the\s+(?:CSC|crop|padding|input)\b.*|Elements\s+in\s+the.*|Output\s+bias\s+for.*|Input\s+bias\s+for.*|"
    r"Mean\s+value\s+of.*|Minimum\s+value\s+of.*|Variance\s+value\s+of.*|For\s+uint8->.*|"
    r"YUV2BGR\s+conversion\s*:.*|BGR2YUV\s+conversion\s*:.*|The\s+computation\s+rules\s+are.*|"
    r"Crop\s+start\s+position.*|Width\s+and\s+height\s+of.*)",
)


def extract_toc_hierarchy_map(text: str, doc_name: str = "") -> dict[str, str]:
    """
    Harvest section number -> 'X.Y Title' mappings from the document's Table of Contents,
    inline chapter navigation lines, and canonical fallback dictionary.
    """
    toc_map = dict(_FALLBACK_SECTION_TITLES.get(doc_name, {}))
    clean = _HTML_TAG_RE.sub(" ", text).replace("**", "")

    # 1. Harvest from pipe ToC rows or dot-leader ToC lines
    for m in re.finditer(
        r"(?:^\|\s*|\b)((?:\d+|[A-Z])(?:\.\d+)*)\s+([A-Za-z][^.|#\n]{2,90}?)\s*(?:\.{2,}|\|\s*\d+\s*\|)",
        clean,
        flags=re.M,
    ):
        sec_num = m.group(1).strip()
        sec_title = re.sub(r"\s+", " ", m.group(2)).strip()
        if sec_num and sec_title and not sec_title.lower().startswith("for details"):
            toc_map[sec_num] = f"{sec_num} {sec_title}"

    # 2. Harvest from inline chapter mini-ToC lines (e.g., "3.1 Specifications [3.2 Environmental...](#page-13-0)")
    for line in clean.splitlines():
        if "#page-" in line and not line.strip().startswith("|") and not line.strip().startswith("-"):
            clean_line = re.sub(r"\[([^\]]+)\]\(#page-[^)]+\)", r"\1", line)
            for m in re.finditer(r"\b(\d+\.\d+(?:\.\d+)*)\s+([A-Z][A-Za-z0-9\s()/\-]+?)(?=\s+\d+\.\d+|\s*$)", clean_line):
                sec_num, sec_title = m.group(1).strip(), m.group(2).strip()
                if sec_num not in toc_map and len(sec_title) > 2:
                    toc_map[sec_num] = f"{sec_num} {sec_title}"

    # 3. Also record any explicit numbered headings in the document body
    for line in clean.splitlines():
        hm = re.match(r"^#{1,6}\s+((?:\d+|[A-Z])(?:\.\d+)*)(?!\.)\s+(.+)$", line.strip())
        if hm:
            sec_num, sec_title = hm.group(1).strip(), hm.group(2).strip()
            if (sec_num.isdigit() or "." in sec_num) and len(sec_title) > 1:
                toc_map[sec_num] = f"{sec_num} {sec_title}"

    return toc_map


def clean_chunk_text(text: str) -> str:
    """Strip residual HTML tags, markdown escapes, bold markers, and fix OCR 'CO' -> 'C0'."""
    text = _HTML_TAG_RE.sub(" ", str(text))
    text = _MD_ESCAPE_RE.sub(r"\1", text)
    text = re.sub(r"(?<![A-Za-z0-9])CO(?=[,/\s\)\].]|$)", "C0", text)
    return text.replace("**", "").strip()


def is_toc_chunk(text: str) -> bool:
    """Return True if a chunk is a Table of Contents (ToC) table with dot leaders or page numbers."""
    table_lines = [l for l in text.splitlines() if l.strip().startswith("|")]
    if len(table_lines) < 2:
        return False
    dot_count = sum(
        1 for l in table_lines
        if re.search(r"\.{5,}", l) or re.search(r"\|\s*\d+(\.\d+)*\s+.+\|\s*\d+\s*\|", l)
    )
    return dot_count >= max(2, len(table_lines) * 0.4)


def _is_pseudo_unnumbered_header(t: str) -> bool:
    """Return True if an unnumbered '#' line is actually a code/AIPP comment, CLI command, or sentence."""
    if not t:
        return True
    if _SUBSECTION_LABELS_RE.match(t) or re.match(r"^(?:Table|Figure)\s+\d+", t, re.I) or re.match(r"^\d+\.\s+", t):
        return True
    if not (t[0].isupper() or t.startswith("(Recommended)") or t.startswith("(Optional)")):
        return True
    if "#" in t or " = " in t or ";" in t or "->" in t or "|" in t or ":" in t:
        return True
    if len(t) > 100 and not (t.startswith("(Recommended)") or t.startswith("(Optional)")):
        return True
    if "." in t and not t.endswith("sess.run()"):
        return True
    return False


def stitch_continuation_tables(text: str, max_merged_rows: int = 80) -> str:
    """
    Merge multi-page pipe tables split across PDF page breaks even when separated
    by blank lines, horizontal rules, or residual running headers/page markers.
    """
    lines = text.splitlines()
    out = []
    i = 0
    sep_re = re.compile(r"^\s*\|(?:\s*:?-+:?\s*\|)+\s*$")
    noise_re = re.compile(
        r"^(?:\s*|\s*---+\s*|\s*\d+\s*|\s*Page\s+\d+.*|"
        r"\s*Huawei\s+Atlas\s+200\s+DK.*|\s*Technical\s+White\s+Paper.*|"
        r"\s*Issue\s+\d+.*|\s*Copyright\s+©.*)$",
        re.I,
    )

    def _is_table_line(ln: str) -> bool:
        s = ln.strip()
        return s.startswith("|") and "|" in s[1:]

    while i < len(lines):
        line = lines[i]
        if _is_table_line(line):
            table_block = [line.rstrip()]
            i += 1
            while i < len(lines) and _is_table_line(lines[i]):
                table_block.append(lines[i].rstrip())
                i += 1

            while True:
                j = i
                # Skip blank lines and page-break running header/footer noise (up to 6 lines)
                skipped = 0
                while j < len(lines) and skipped < 6 and not _is_table_line(lines[j]):
                    if noise_re.match(lines[j]):
                        j += 1
                        skipped += 1
                    else:
                        break

                if j < len(lines) and _is_table_line(lines[j]):
                    next_block = []
                    k = j
                    while k < len(lines) and _is_table_line(lines[k]):
                        next_block.append(lines[k].rstrip())
                        k += 1

                    cols_1 = table_block[0].strip().count("|")
                    cols_2 = next_block[0].strip().count("|")
                    if cols_1 == cols_2 and (len(table_block) + len(next_block) <= max_merged_rows):
                        if (
                            len(next_block) >= 2
                            and sep_re.match(next_block[1].strip())
                            and next_block[0].strip().lower() == table_block[0].strip().lower()
                        ):
                            table_block.extend(next_block[2:])
                        elif len(next_block) >= 1 and sep_re.match(next_block[0].strip()):
                            table_block.extend(next_block[1:])
                        else:
                            table_block.extend(next_block)
                        i = k
                        continue
                break

            out.extend(table_block)
        else:
            out.append(line)
            i += 1

    return "\n".join(out)


def postprocess_markdown(text: str, doc_name: str = "") -> str:
    """
    Apply Huawei-specific structural and hierarchy normalization to extracted Markdown:
    1. Strip HTML spans, image placeholders, and proprietary running headers/footers.
    2. Rejoin hyphenated technical terms, normalize hardware units, and fix OCR 'CO' -> 'C0'.
    3. Harvest the document's Table of Contents to restore missing top-level chapter headings.
    4. Normalize numbered section headings (#, ##, ###, ####) and demote callouts/comments.
    5. Stitch multi-page continuation tables split across PDF page breaks.
    """
    toc_map = extract_toc_hierarchy_map(text, doc_name=doc_name)

    # 1. Strip HTML tags, markdown escapes, and proprietary headers/footers
    text = _HTML_TAG_RE.sub(" ", text)
    text = _MD_ESCAPE_RE.sub(r"\1", text)
    for pat in _HEADER_FOOTER_PATTERNS:
        text = pat.sub("", text)

    # 2. Fix broken technical terms and OCR 'CO' -> 'C0'
    for pat, repl in _HYPHEN_REJOINS:
        text = pat.sub(repl, text)
    for pat, repl in _UNIT_NORMALIZATIONS:
        text = pat.sub(repl, text)
    text = re.sub(r"(?<![A-Za-z0-9])CO(?=[,/\s\)\].]|$)", "C0", text)

    # 3. Demote callouts, Figure/Table pseudo-headers, numbered list items, and AIPP config comments
    lines = text.splitlines()
    norm_lines = []
    seen_headings = set()

    for line in lines:
        stripped = line.strip()

        # Strip inline chapter mini-ToC navigation lines (e.g. "1.1 Overview [1.2 Appearance](#page-6-0)")
        if (
            not stripped.startswith("#")
            and not stripped.startswith("|")
            and not stripped.startswith("-")
            and re.match(r"^\d+\.\d+\s+[A-Z].*(?:\[\d+\.\d+|\d+\.\d+\s+[A-Z])", stripped)
            and ("#page-" in stripped or len(re.findall(r"\b\d+\.\d+\b", stripped)) >= 2)
        ):
            continue

        # Demote AIPP config comment lines that start with '# '
        if _CONFIG_COMMENT_HEADER_RE.match(stripped):
            comment_body = re.sub(r"^#{1,6}\s*", "", stripped).replace("**", "")
            norm_lines.append(f"// {comment_body}")
            continue

        hm = re.match(r"^(#{1,6})\s+(.+)$", stripped)
        if hm:
            raw_title = hm.group(2).replace("**", "").strip()
            raw_title = re.sub(r"\s+", " ", raw_title)

            # Demote callout boxes (NOTE, NOTICE, CAUTION, WARNING)
            if re.match(r"^(?:NO\s*TE|NO\s*TICE|CAUTION|WARNING|DANGER|TIP)\s*$", raw_title, re.I):
                label = raw_title.replace(" ", "").upper()
                norm_lines.append(f"**{label}:**")
                continue

            # Check if this is a numbered section heading (e.g., "1", "1.1", "4.3.3", "A.1")
            num_m = re.match(r"^((?:\d+|[A-Z])(?:\.\d+)*)(?!\.)\s+(.+)$", raw_title)
            if num_m and (num_m.group(1).isdigit() or "." in num_m.group(1)):
                sec_num = num_m.group(1)
                parts = sec_num.split(".")
                # Inject any missing parent headings (e.g. Chapter '5' before '5.1')
                for depth in range(1, len(parts)):
                    parent_num = ".".join(parts[:depth])
                    if parent_num not in seen_headings and parent_num in toc_map:
                        parent_hashes = "#" * min(depth, 4)
                        norm_lines.append(f"{parent_hashes} {toc_map[parent_num]}")
                        norm_lines.append("")
                        seen_headings.add(parent_num)

                seen_headings.add(sec_num)
                hashes = "#" * min(len(parts), 4)
                norm_lines.append(f"{hashes} {raw_title}")
                continue

            # Unnumbered heading: demote if it is a pseudo-header (comment, CLI command, label, caption)
            if _is_pseudo_unnumbered_header(raw_title):
                norm_lines.append(f"**{raw_title}**")
                continue

        norm_lines.append(line)

    text = "\n".join(norm_lines)
    text = stitch_continuation_tables(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# ============================================================================
# Strategy 3: Atomic Table-Aware & Breadcrumb Hybrid Chunking
# ============================================================================

def strategy_atomic_table_aware(
    text: str,
    doc_name: str = "unknown",
    max_chunk_tokens: int = 1024,
    chunk_overlap: int = 96,
    min_chunk_tokens: int = 220,
) -> list[dict]:
    """
    Strategy 3 (Enhanced): Atomic table-aware chunking with budget-aware section packing,
    bidirectional table context (pre-caption + post-table notes/examples), and code preservation.

    Pipeline:
    1. Harvest canonical chapter/section titles from the document's Table of Contents.
    2. Demote callouts (NOTE/NOTICE), captions (Table/Figure), numbered list items ("4. Concat"),
       CLI commands, and AIPP/Python '#' comment lines so they never corrupt section trails.
    3. Track numbered hierarchy (levels 1..4) separately from unnumbered subheaders so sibling
       unnumbered headers replace each other at the same depth rather than chaining indefinitely,
       and validate major chapter consistency across subsections.
    4. Merge bare heading stubs and sibling unnumbered micro-sections while keeping distinct
       numbered specification sections separate.
    5. Keep pipe tables and fenced code blocks atomic, prepending up to 3 lines of preceding
       prose (table title/caption) to each table chunk.
    """
    toc_map = extract_toc_hierarchy_map(text, doc_name=doc_name)

    # 1. Pre-clean callout boxes and table/figure pseudo-headers
    text = re.sub(
        r"^#{1,6}\s*\*?\*?(NO\s*TE|NO\s*TICE|CAUTION|WARNING|DANGER|TIP)\*?\*?\s*$",
        lambda m: f"**{m.group(1).replace(' ', '')}:**",
        text,
        flags=re.MULTILINE | re.IGNORECASE,
    )
    text = re.sub(
        r"^#{1,6}\s*\*?\*?(Table\s+\d+[-\d.]*|Figure\s+\d+[-\d.]*)\*?\*?",
        r"**\1**",
        text,
        flags=re.MULTILINE | re.IGNORECASE,
    )
    text = stitch_continuation_tables(text)

    lines = text.split("\n")
    raw_sections = []
    numbered_headers: dict[int, str] = {}
    unnumbered_sub: Optional[str] = None
    current_lines: list[str] = []
    current_has_table = False

    def _current_path() -> str:
        parts = [numbered_headers[k] for k in sorted(numbered_headers.keys())]
        if unnumbered_sub:
            parts.append(unnumbered_sub)
        return " > ".join(parts) if parts else "Root"

    def _numbered_base_path() -> str:
        parts = [numbered_headers[k] for k in sorted(numbered_headers.keys())]
        return " > ".join(parts) if parts else "Root"

    def _flush_section():
        nonlocal current_lines, current_has_table
        body = "\n".join(current_lines).strip()
        if body:
            raw_sections.append((
                _current_path(),
                _numbered_base_path(),
                unnumbered_sub is not None,
                body,
                current_has_table,
            ))
        current_lines = []
        current_has_table = False

    for line in lines:
        stripped = line.strip()

        # Skip inline chapter mini-ToC navigation lines
        if (
            not stripped.startswith("#")
            and not stripped.startswith("|")
            and not stripped.startswith("-")
            and re.match(r"^\d+\.\d+\s+[A-Z].*(?:\[\d+\.\d+|\d+\.\d+\s+[A-Z])", stripped)
            and ("#page-" in stripped or len(re.findall(r"\b\d+\.\d+\b", stripped)) >= 2)
        ):
            continue

        # Protect AIPP/code comment lines starting with '# ' from being parsed as headers
        if _CONFIG_COMMENT_HEADER_RE.match(stripped):
            current_lines.append(re.sub(r"^#{1,6}\s*", "// ", stripped).replace("**", ""))
            continue

        header_match = re.match(r"^(#{1,6})\s+(.+)$", stripped)
        if header_match:
            raw_title = _HTML_TAG_RE.sub(" ", header_match.group(2)).replace("**", "").strip()
            raw_title = re.sub(r"\s+", " ", raw_title)

            num_match = re.match(r"^((?:\d+|[A-Z])(?:\.\d+)*)(?!\.)\s+(.+)$", raw_title)
            is_valid_num = bool(num_match and (num_match.group(1).isdigit() or "." in num_match.group(1)))

            if not is_valid_num and _is_pseudo_unnumbered_header(raw_title):
                current_lines.append(f"**{raw_title}**")
                continue

            _flush_section()

            if is_valid_num:
                sec_num = num_match.group(1)
                parts = sec_num.split(".")
                level = min(len(parts), 4)
                major_chap = parts[0]

                if level >= 2:
                    cur_l1 = numbered_headers.get(1, "")
                    if not cur_l1.startswith(f"{major_chap} "):
                        numbered_headers.clear()
                        if major_chap in toc_map:
                            numbered_headers[1] = toc_map[major_chap]

                    for d in range(2, level):
                        prefix = ".".join(parts[:d])
                        if d not in numbered_headers or not numbered_headers[d].startswith(f"{prefix} "):
                            if prefix in toc_map:
                                numbered_headers[d] = toc_map[prefix]

                numbered_headers[level] = raw_title
                for k in list(numbered_headers.keys()):
                    if k > level:
                        del numbered_headers[k]
                unnumbered_sub = None
            else:
                if not numbered_headers:
                    numbered_headers[1] = raw_title
                    unnumbered_sub = None
                else:
                    unnumbered_sub = raw_title

            current_lines.append(f"## {raw_title}")
        else:
            current_lines.append(line)
            if stripped.startswith("|") and "|" in stripped[1:]:
                current_has_table = True

    _flush_section()

    # 2. Smart Micro-Section Merging (including sections with small tables if within token budget)
    sections = []
    for hp, num_base, is_unnum, st, ht in raw_sections:
        if sections:
            prev_hp, prev_num_base, prev_is_unnum, prev_st, prev_ht = sections[-1]
            prev_tok = count_tokens(prev_st)
            cur_tok = count_tokens(st)

            # Case A: Bare heading stub (< 40 tokens) that is a prefix of current section
            if prev_tok < 40 and hp.startswith(prev_hp + " > "):
                sections[-1] = (hp, num_base, is_unnum, f"{prev_st}\n\n{st}", ht or prev_ht)
                continue

            # Case B: Sibling unnumbered sub-sections under the SAME numbered section
            if (
                num_base != "Root"
                and num_base == prev_num_base
                and (is_unnum or prev_is_unnum)
                and (prev_tok < min_chunk_tokens or cur_tok < min_chunk_tokens)
                and (prev_tok + cur_tok <= max_chunk_tokens - 40)
            ):
                merged_hp = prev_hp if prev_hp == hp else f"{prev_hp} / {hp.split(' > ')[-1]}"
                sections[-1] = (merged_hp, num_base, True, f"{prev_st}\n\n{st}", ht or prev_ht)
                continue

        sections.append((hp, num_base, is_unnum, st, ht))

    # 3. Budget-Aware Emission:
    #    - Keep entire section intact if it fits within max_chunk_tokens (prevents divorcing
    #      tables/code from post-table notes, examples, and parameter descriptions).
    #    - Only split on pipe tables when the section exceeds max_chunk_tokens, attaching
    #      BOTH preceding caption lines and immediate trailing notes/examples to each table.
    results = []
    table_split_re = re.compile(r"((?:^\s*\|.+\|\s*$\n?){2,})", flags=re.MULTILINE)
    sub_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_tokens,
        chunk_overlap=chunk_overlap,
        length_function=count_tokens,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    for header_path, _, _, section_text, has_table in sections:
        breadcrumb = f"[Doc: {doc_name} > {header_path}]"
        full_section_chunk = f"{breadcrumb}\n\n{section_text.strip()}"
        full_tok_count = count_tokens(full_section_chunk)

        # Fast path: entire section (prose + code + tables) fits in a single chunk
        if full_tok_count <= max_chunk_tokens:
            results.append({
                "text": full_section_chunk,
                "strategy": "atomic_table_aware",
                "chunk_index": len(results),
                "approx_tokens": full_tok_count,
                "metadata": {
                    "doc_name": doc_name,
                    "header_path": header_path,
                    "is_table": bool(has_table),
                    "is_code_block": "```" in section_text,
                },
            })
            continue

        # Section exceeds max_chunk_tokens: split only around pipe tables (keep code with prose)
        raw_parts = [p.strip() for p in table_split_re.split(section_text) if p and p.strip()]

        for idx, part_stripped in enumerate(raw_parts):
            is_table = part_stripped.startswith("|")

            if is_table:
                # Attach up to 4 preceding prose lines (caption/intro) and up to 4 trailing prose lines (notes/examples)
                prefix_lines = []
                if idx > 0 and not raw_parts[idx - 1].startswith("|"):
                    prev_non_empty = [ln.strip() for ln in raw_parts[idx - 1].splitlines() if ln.strip()]
                    prefix_lines = prev_non_empty[-4:]

                suffix_lines = []
                if idx + 1 < len(raw_parts) and not raw_parts[idx + 1].startswith("|"):
                    next_non_empty = [
                        ln.strip() for ln in raw_parts[idx + 1].splitlines()
                        if ln.strip() and not ln.strip().startswith("## ")
                    ]
                    suffix_lines = next_non_empty[:4]

                assembled = part_stripped
                if prefix_lines:
                    assembled = "\n".join(prefix_lines) + "\n\n" + assembled
                if suffix_lines:
                    assembled = assembled + "\n\n" + "\n".join(suffix_lines)

                chunk_text = f"{breadcrumb}\n\n{assembled}"
                results.append({
                    "text": chunk_text,
                    "strategy": "atomic_table_aware",
                    "chunk_index": len(results),
                    "approx_tokens": count_tokens(chunk_text),
                    "metadata": {
                        "doc_name": doc_name,
                        "header_path": header_path,
                        "is_table": True,
                        "is_code_block": False,
                    },
                })
            else:
                chunk_text = f"{breadcrumb}\n\n{part_stripped}"
                tok_count = count_tokens(chunk_text)
                if tok_count <= max_chunk_tokens:
                    results.append({
                        "text": chunk_text,
                        "strategy": "atomic_table_aware",
                        "chunk_index": len(results),
                        "approx_tokens": tok_count,
                        "metadata": {
                            "doc_name": doc_name,
                            "header_path": header_path,
                            "is_table": False,
                            "is_code_block": "```" in part_stripped,
                        },
                    })
                else:
                    for sc in sub_splitter.split_text(part_stripped):
                        sc_with_bc = f"{breadcrumb}\n\n{sc}"
                        results.append({
                            "text": sc_with_bc,
                            "strategy": "atomic_table_aware",
                            "chunk_index": len(results),
                            "approx_tokens": count_tokens(sc_with_bc),
                            "metadata": {
                                "doc_name": doc_name,
                                "header_path": header_path,
                                "is_table": False,
                                "is_code_block": "```" in sc,
                            },
                        })

    return results


print("[OK] Strategy 3 (Atomic Table-Aware) defined")

# ============================================================================
# Strategy 1: Recursive Character Splitting (COMMENTED FOR FUTURE BENCHMARKING)
# ============================================================================
# def strategy_recursive_split(
#     text: str,
#     doc_name: str = "unknown",
#     chunk_size: int = 1024,
#     chunk_overlap: int = 128,
# ) -> list[dict]:
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         length_function=count_tokens,
#         separators=["\n\n", "\n", ". ", " ", ""],
#     )
#     chunks = splitter.split_text(text)
#     return [
#         {
#             "text": f"[Doc: {doc_name}]\n\n{chunk}",
#             "strategy": "recursive",
#             "chunk_index": i,
#             "approx_tokens": count_tokens(chunk),
#             "metadata": {"doc_name": doc_name, "header_path": "Recursive", "is_table": "|" in chunk},
#         }
#         for i, chunk in enumerate(chunks)
#     ]

[OK] Strategy 3 (Atomic Table-Aware) defined


# Carregamento dos Manuais e Padronização dos Chunks

Esta célula particiona os 10 manuais Markdown utilizando a **Estratégia 3 (Atomic Table-Aware)**. Realiza a limpeza de tags HTML/escapes, correção de OCR em `C0`, exclusão de chunks de sumário (ToC) e preservação atômica das tabelas com legendas associadas.

In [52]:
LOAD_PRECOMPUTED_CHUNKS = False

rows = []
print(f"[INFO] Computing atomic table-aware chunks (Strategy 3) from {len(md_paths)} Markdown manuals...")
for md_path in md_paths:
    doc_slug = re.sub(r"[^\w\-.]", "_", md_path.stem.strip())
    raw_text = md_path.read_text(encoding="utf-8")
    norm_doc_text = postprocess_markdown(raw_text, doc_name=md_path.stem)
    cleaned_doc_text = clean_chunk_text(norm_doc_text)
    chunks = strategy_atomic_table_aware(cleaned_doc_text, doc_name=md_path.stem)
    for i, c in enumerate(chunks):
        rows.append({
            "id": f"{doc_slug}_{i:04d}",
            "text": clean_chunk_text(c["text"]),
            "doc": md_path.stem,
            "chunk": i,
            "section": clean_chunk_text(c["metadata"]["header_path"]),
            "is_table": c["metadata"]["is_table"],
        })

print(f"[OK] Generated {len(rows):,} raw atomic chunks from {len(md_paths)} documents.")

# Filter out Table of Contents (ToC) chunks
filtered_rows = []
toc_count = 0
for r in rows:
    if is_toc_chunk(r["text"]):
        toc_count += 1
    else:
        filtered_rows.append(r)
rows = filtered_rows
print(f"[INFO] Filtered out {toc_count:,} Table of Contents (ToC) noise chunks. Remaining: {len(rows):,}")

table_chunks = sum(1 for r in rows if r["is_table"])
print(f"[SUMMARY] Total chunks: {len(rows):,} | Specification table chunks: {table_chunks:,} ({table_chunks / len(rows):.1%})")

[INFO] Computing atomic table-aware chunks (Strategy 3) from 10 Markdown manuals...
[OK] Generated 1,406 raw atomic chunks from 10 documents.
[INFO] Filtered out 17 Table of Contents (ToC) noise chunks. Remaining: 1,389
[SUMMARY] Total chunks: 1,389 | Specification table chunks: 485 (34.9%)


# Indexação Híbrida: Vetorial (ChromaDB + nomic-embed-text) e Léxica (Okapi BM25)

Criação da coleção vetorial persistente `"huawei_ascend_atomic_v3_col"` usando embeddings densos do modelo `nomic-embed-text` com prefixação assimétrica (`search_document: ` para indexação e `search_query: ` para consultas), integrada a um índice léxico Okapi BM25 em memória com fusão de rankings via Reciprocal Rank Fusion (RRF).

In [53]:
_LEX_TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[_\-.][A-Za-z0-9]+)*")
_CAMEL_RE = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z0-9]+|[A-Z0-9]+")
_STOPWORDS = set(
    "a an the of to in on for and or is are was were be been being what which who whom how why when where "
    "does do did can could should would will shall may might must this that these those it its with by from "
    "at as into about than then there their them they i my we our you your if not no so such any all each "
    "use used using via per doc section".split()
)

def lexical_tokenize(text: str) -> list[str]:
    out = []
    for m in _LEX_TOKEN_RE.findall(text):
        low = m.lower()
        if low not in _STOPWORDS:
            out.append(low)
        parts = [p for p in re.split(r"[_\-.]", m) if p]
        subs = []
        for p in parts:
            subs.extend(_CAMEL_RE.findall(p) if re.search(r"[a-z][A-Z]", p) else [p])
        if len(subs) > 1:
            out.extend(s.lower() for s in subs if s.lower() not in _STOPWORDS and len(s) > 1)
    return out

class BM25Index:
    """Lightweight Okapi BM25 index over rows for hybrid lexical + dense retrieval."""
    def __init__(self, rows: list[dict], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.rows = rows
        self.id_to_row = {r["id"]: r for r in rows}
        docs_tokens = [lexical_tokenize(r["text"]) for r in rows]
        self.n_docs = len(docs_tokens)
        self.lens = np.array([max(1, len(t)) for t in docs_tokens], dtype=float)
        self.avg_len = float(self.lens.mean()) if self.n_docs else 1.0
        self.postings = defaultdict(list)
        for idx, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((idx, tf))
        self.idf = {
            t: math.log(1.0 + (self.n_docs - len(p) + 0.5) / (len(p) + 0.5))
            for t, p in self.postings.items()
        }

    def search_ids(self, query: str, top_n: int = 15) -> list[str]:
        scores = np.zeros(self.n_docs, dtype=float)
        for term in set(lexical_tokenize(query)):
            if term not in self.postings:
                continue
            idf = self.idf[term]
            for idx, tf in self.postings[term]:
                norm = self.k1 * (1.0 - self.b + self.b * self.lens[idx] / self.avg_len)
                scores[idx] += idf * tf * (self.k1 + 1.0) / (tf + norm)
        top_indices = np.argsort(-scores)[:top_n]
        return [self.rows[i]["id"] for i in top_indices if scores[i] > 0]

BM25_INDEX = BM25Index(rows)
print(f"[OK] BM25 lexical index built over {len(rows):,} chunks ({len(BM25_INDEX.idf):,} unique terms).")

def embed(texts, prefix="search_document: ", model=EMB_MODEL):
    """Obtain dense embeddings via Ollama REST API with asymmetric prefixing."""
    prefixed = [f"{prefix}{t}" for t in texts]
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": prefixed})
    r.raise_for_status()
    return r.json()["embeddings"]

def build_index(rows, name=COLLECTION_NAME, persist_dir=CHROMA_DIR, batch=64, rebuild=False):
    """Build or load a ChromaDB PersistentClient index with batching and progress tracking."""
    store = chromadb.PersistentClient(path=persist_dir)

    if rebuild:
        try:
            store.delete_collection(name)
            print(f"[INFO] Deleted existing collection: {name}")
        except Exception:
            pass

    col = store.get_or_create_collection(
        name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL},
    )

    current_count = col.count()
    if current_count == len(rows):
        print(f"[OK] Collection '{name}' already indexed with {current_count:,} chunks. Skipping re-indexing.")
        return col

    print(f"[INFO] Indexing {len(rows):,} chunks into '{name}' (batch size = {batch})...")
    for i in tqdm(range(0, len(rows), batch), desc="Indexing Chunks", unit="batch"):
        part = rows[i : i + batch]
        col.upsert(
            ids=[r["id"] for r in part],
            documents=[r["text"] for r in part],
            embeddings=embed([r["text"] for r in part], prefix="search_document: "),
            metadatas=[
                {
                    "doc": r["doc"],
                    "chunk": r["chunk"],
                    "section": r["section"],
                    "is_table": r["is_table"],
                }
                for r in part
            ],
        )

    print(f"[OK] Successfully built index! Total items in collection: {col.count():,}")
    return col

def retrieve(col, question, k=5, dense_candidates=15, bm25_candidates=15, rrf_c=60):
    """Hybrid retrieval: fuses ChromaDB dense cosine search with BM25 lexical search via RRF."""
    assert col.metadata.get("embedding_model") == EMB_MODEL, "Embedding model mismatch"
    res = col.query(
        query_embeddings=embed([question], prefix="search_query: "),
        n_results=dense_candidates,
        include=["documents", "metadatas", "distances"],
    )
    dense_hits = {}
    dense_ranking = []
    for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        dense_ranking.append(i)
        dense_hits[i] = {
            "id": i,
            "text": t,
            "doc": m["doc"],
            "section": m["section"],
            "is_table": m.get("is_table", False),
            "distance": float(d),
        }

    bm25_ranking = BM25_INDEX.search_ids(question, top_n=bm25_candidates)

    # Reciprocal Rank Fusion (Dense + BM25)
    fused_scores = defaultdict(float)
    for rank, cid in enumerate(dense_ranking, 1):
        fused_scores[cid] += 1.0 / (rrf_c + rank)
    for rank, cid in enumerate(bm25_ranking, 1):
        fused_scores[cid] += 1.0 / (rrf_c + rank)

    sorted_cids = [cid for cid, _ in sorted(fused_scores.items(), key=lambda x: -x[1])[:k]]
    final_hits = []
    for cid in sorted_cids:
        if cid in dense_hits:
            final_hits.append(dense_hits[cid])
        else:
            r = BM25_INDEX.id_to_row[cid]
            final_hits.append({
                "id": cid,
                "text": r["text"],
                "doc": r["doc"],
                "section": r["section"],
                "is_table": r["is_table"],
                "distance": 0.35,  # fallback distance when surfaced purely by BM25
            })
    return final_hits

def sources_used(hits):
    """Format unique citations as [<doc> · <section>]."""
    seen, out = set(), []
    for h in hits:
        key = f"[{h['doc']} · {h['section']}]"
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

[OK] BM25 lexical index built over 1,389 chunks (15,014 unique terms).


In [54]:
col = build_index(rows, name=COLLECTION_NAME, batch=128, rebuild=True)

[INFO] Deleted existing collection: huawei_ascend_atomic_v3_col
[INFO] Indexing 1,389 chunks into 'huawei_ascend_atomic_v3_col' (batch size = 128)...


Indexing Chunks:   0%|          | 0/11 [00:00<?, ?batch/s]

[OK] Successfully built index! Total items in collection: 1,389


# Motor de Geração RAG e Formatação de Contexto

Configuração do prompt de sistema adaptado ao ecossistema Huawei Ascend e função `ask_rag()` que orquestra a recuperação híbrida (k=5), montagem de contexto estruturado e chamada do modelo `qwen2.5:3b`.

In [55]:
SYSTEM_ANSWER = (
    "You are a technical assistant specializing in Huawei AI hardware (Ascend 310, Atlas 200 DK) "
    "and the CANN/DDK software ecosystem (ATC, OMG, AscendCL, AIPP, TE/TVM).\n"
    "Answer the user's question strictly using the context provided below. "
    "Be precise with parameter names, compilation flags, data types, and hardware specifications.\n"
    "If the information is not present in the context, objectively declare that you do not have sufficient information."
)

def format_context(hits):
    """Assemble formatted retrieved chunks with document header metadata."""
    parts = []
    for h in hits:
        table_tag = " [Table Chunk]" if h.get("is_table") else ""
        parts.append(f"Document: {h['doc']} (Section: {h['section']}){table_tag}\nContent:\n{h['text']}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, k=5):
    """Full RAG loop: dense retrieval + context formatting + LLM generation."""
    t0 = time.time()
    hits = retrieve(col, question, k=k)
    t_retrieval = time.time() - t0

    context = format_context(hits)
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    t1 = time.time()
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_ANSWER},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
    )
    t_gen = time.time() - t1

    answer = response.choices[0].message.content
    sources = sources_used(hits)

    return {
        "answer": answer,
        "sources": sources,
        "hits": hits,
        "latency_retrieval_s": round(t_retrieval, 3),
        "latency_gen_s": round(t_gen, 3),
        "latency_total_s": round(t_retrieval + t_gen, 3),
    }

# Validação com Suíte Smoke-Test (5 Consultas Técnicas de Domínio)

Execução de 5 consultas técnicas representativas cobrindo os pilares do ecossistema Huawei Ascend para inspecionar a qualidade de recuperação híbrida (k=5), presença de tabelas e respostas do LLM:
1. **Exact Flag / Compiler:** `--input_fp16_nodes` no ATC
2. **Hardware Revisions / Specs:** Diferenças entre IT21DMDA e IT21VDMB
3. **AIPP Preprocessing:** Coeficientes da matriz CSC YUV420SP -> RGB888
4. **Troubleshooting / System Error:** Erro `CheckPartitionSpace partition space check failed`
5. **Tensor Architecture / Memory Layout:** Formato 5D NC1HWC0 e significado de C0

In [56]:
SMOKE_TEST_QUERIES = [
    {
        "id": "ST1",
        "domain": "Exact Flag / Compiler",
        "target_doc": "ATCToolInstructions.md",
        "query": "What does the --input_fp16_nodes parameter do in the ATC tool?",
    },
    {
        "id": "ST2",
        "domain": "Hardware Revisions / Specs",
        "target_doc": "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
        "query": "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
    },
    {
        "id": "ST3",
        "domain": "AIPP Preprocessing",
        "target_doc": "ATCToolInstructions.md / Atlas200AIAcceleratorModule...",
        "query": "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
    },
    {
        "id": "ST4",
        "domain": "Troubleshooting / System Error",
        "target_doc": "EnvironmentDeploymentGuide.md",
        "query": "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade and how to fix it?",
    },
    {
        "id": "ST5",
        "domain": "Tensor Architecture / Memory Layout",
        "target_doc": "ATCToolInstructions.md / TBECustomOperator...",
        "query": "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16 and INT8?",
    },
]

print(f"Running smoke test on {len(SMOKE_TEST_QUERIES)} Huawei Ascend domain queries...\n")

for item in SMOKE_TEST_QUERIES:
    print("=" * 90)
    print(f"[{item['id']}] Domain: {item['domain']}")
    print(f"Target Doc: {item['target_doc']}")
    print(f"Query: {item['query']}")
    print("-" * 90)

    res = ask_rag(item["query"], k=5)

    print(f"Answer:\n{res['answer']}\n")
    print(f"Latencies: Retrieval={res['latency_retrieval_s']}s | Generation={res['latency_gen_s']}s | Total={res['latency_total_s']}s")
    print("Retrieved Sources:")
    for h in res["hits"]:
        tbl_info = " [TABLE]" if h.get("is_table") else ""
        print(f"  - doc={h['doc']} | dist={h['distance']:.4f}{tbl_info} | section={h['section']}")
    print()

Running smoke test on 5 Huawei Ascend domain queries...

[ST1] Domain: Exact Flag / Compiler
Target Doc: ATCToolInstructions.md
Query: What does the --input_fp16_nodes parameter do in the ATC tool?
------------------------------------------------------------------------------------------
Answer:
The `--input_fp16_nodes` parameter in the ATC (Ascend Tool Command) is used to specify the names of input nodes that have data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`.

When you use this parameter, you need to provide a string that lists the names of the input nodes, each enclosed in double quotation marks and separated by semicolons. For example:

```sh
atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes="data1;data2"
```

In this example, `data1` and `data2` are the names of the input nodes that are expected to have FP16 data t

# Técnicas de Pré-Consulta para Otimização do RAG

Implementação das 4 variantes de recuperação pré-consulta para avaliação comparativa:
- **Baseline:** Busca direta pelo texto original da pergunta via recuperação híbrida.
- **Step-Back Prompting:** Abstração da pergunta para recuperar conceitos fundamentais de alto nível.
- **Multi-Query / RAG-Fusion:** Geração de múltiplas reformulações com fusão de rankings via Reciprocal Rank Fusion (RRF).
- **RAG com Memória:** Reescreve perguntas dependentes de contexto histórico em consultas completas e independentes.

In [57]:
# ============== PARAMETERS CONFIGURATION ==============
K            = 4      # chunks entregues ao gerador
N_RUNS       = 1      # execuções por variante
SEED_BASE    = 42     # seed para reprodutibilidade
GEN_TEMP     = 0.5    # temperatura da resposta final
REWRITE_TEMP = 0.3    # temperatura das chamadas de reescrita
N_VARIANTS   = 3      # reformulações no Multi-Query
RRF_C        = 15     # constante do Reciprocal Rank Fusion
MEMORY_DISTRACTOR = True
FORCE        = True
# =======================================================

class Trace:
    def __init__(self):
        self.llm_calls = 0
        self.embed_calls = 0
        self.queries = []

def llm(messages, tr, temperature, seed):
    tr.llm_calls += 1
    r = client.chat.completions.create(model=GEN_MODEL, messages=messages,
                                       temperature=temperature, seed=seed)
    return r.choices[0].message.content.strip()

def search(col, query, tr, k=K):
    tr.embed_calls += 1
    tr.queries.append(query)
    return retrieve(col, query, k=k)

def rrf(rankings, c=RRF_C):
    """Reciprocal Rank Fusion"""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking, 1):
            fused[cid] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

def dedupe_topk(hits, k=K):
    best = {}
    for idx, h in enumerate(hits):
        # Preserve hybrid RRF rank order while deduplicating
        if h["id"] not in best:
            best[h["id"]] = h
    return list(best.values())[:k]

def clean_lines(text, original, n):
    out = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"')
        if line and line.lower() != original.lower() and len(line) < 300 and line not in out:
            out.append(line)
    return out[:n]

def generate_answer(question, hits, tr, seed):
    messages = [
        {"role": "system", "content": SYSTEM_ANSWER},
        {"role": "user", "content": f"Context:\n{format_context(hits)}\n\nQuestion: {question}"},
    ]
    return llm(messages, tr, GEN_TEMP, seed)

def r_baseline(q, col, tr, seed):
    return q["question"], search(col, q["question"], tr)

### Step-Back Prompting (Abstração de Conceito)

In [58]:
STEPBACK_SYSTEM = (
    "You reformulate questions for searching Huawei AI accelerator technical documentation (Ascend 310, Atlas 200 DK, CANN, ATC). "
    "Given a specific question, write ONE broader, more general (step-back) question about the underlying concept, architecture, parameter, or technical workflow. "
    "Reply only with the general question.\n\n"
    "Example 1\nQuestion: What does the --input_fp16_nodes flag do during model conversion with ATC?\n"
    "General question: What are the precision control parameters and supported data types of the ATC compiler?\n\n"
    "Example 2\nQuestion: How do I configure matrix_r0c0 to convert YUV420SP to RGB888 in the AIPP configuration file?\n"
    "General question: How does color space conversion (CSC) work in the AIPP preprocessing module?\n\n"
    "Example 3\nQuestion: Why does the CheckPartitionSpace error occur when upgrading the driver on the Atlas 200 DK board?\n"
    "General question: What are the partition requirements and driver upgrade procedures on the Atlas 200 DK?"
)

def r_stepback(q, col, tr, seed):
    out = llm([{"role": "system", "content": STEPBACK_SYSTEM},
               {"role": "user", "content": f"Question: {q['question']}\nGeneral question:"}],
              tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    step_q = lines[0] if lines else q["question"]
    hits = search(col, q["question"], tr) + search(col, step_q, tr)
    return q["question"], dedupe_topk(hits, K)

### Multi-Query / RAG-Fusion (Reformulação com Sinônimos)

In [59]:
def r_multiquery(q, col, tr, seed):
    system = (
        f"Generate {N_VARIANTS} different technical reformulations of the user's question for searching Ascend and CANN hardware manuals, "
        "using synonyms, alternative phrasing, and technical variations. "
        "Write one per line, without numbering or explanations."
    )
    out = llm([{"role": "system", "content": system}, {"role": "user", "content": q["question"]}],
              tr, REWRITE_TEMP, seed)
    queries = [q["question"]] + clean_lines(out, q["question"], N_VARIANTS)
    rankings, pool = [], {}
    for qq in queries:
        hs = search(col, qq, tr)
        rankings.append([h["id"] for h in hs])
        for h in hs:
            if h["id"] not in pool or h["distance"] < pool[h["id"]]["distance"]:
                pool[h["id"]] = h
    hits = []
    for cid, score in rrf(rankings)[:K]:
        h = dict(pool[cid])
        h["rrf"] = round(score, 5)
        hits.append(h)
    return q["question"], hits

### RAG com Memória (Perguntas de Acompanhamento)

In [60]:
DISTRACTOR = [
    {
        "role": "user",
        "content": "What is the INT8 computing power of the Ascend 310 Processor?",
    },
    {
        "role": "assistant",
        "content": "The Ascend 310 Processor delivers up to 22 TOPS in INT8 with a typical power consumption of 8W.",
    },
]

MEMORY_SYSTEM = (
    "Given the conversation history and a follow-up question about Huawei Ascend hardware and software, "
    "rewrite the follow-up question so that it is standalone and self-contained. "
    "If it is already standalone, repeat it without changes. Reply only with the rewritten question."
)

def r_memory(q, col, tr, seed):
    history = q.get("history") or (DISTRACTOR if MEMORY_DISTRACTOR else [])
    if not history:
        return q["question"], search(col, q["question"], tr)
    hist = "\n".join(f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in history[-4:])
    out = llm([{"role": "system", "content": MEMORY_SYSTEM},
               {"role": "user", "content": f"History:\n{hist}\n\nFollow-up question: {q['question']}\n\nStandalone question:"}],
               tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    standalone = lines[0] if lines else q["question"]
    return standalone, search(col, standalone, tr)

# VARIANTS = {"baseline": r_baseline, "stepback": r_stepback, "multiquery": r_multiquery, "memory": r_memory}

### Combinando Variantes

In [61]:
def r_combined(q, col, tr, seed):
    # 1. Memory: Resolve histórico (apenas quando houver histórico real na questão)
    history = q.get("history") or []
    if history:
        hist = "\n".join(f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in history[-4:])
        out_mem = llm([
            {"role": "system", "content": MEMORY_SYSTEM},
            {"role": "user", "content": f"History:\n{hist}\n\nFollow-up question: {q['question']}\n\nStandalone question:"}
        ], tr, REWRITE_TEMP, seed)
        lines_mem = [l.strip() for l in out_mem.splitlines() if l.strip()]
        standalone_q = lines_mem[0] if lines_mem else q["question"]
    else:
        standalone_q = q["question"]

    # 2. Step-Back: Gera uma pergunta conceitual ampla a partir da standalone_q
    out_sb = llm([
        {"role": "system", "content": STEPBACK_SYSTEM},
        {"role": "user", "content": f"Question: {standalone_q}\nGeneral question:"}
    ], tr, REWRITE_TEMP, seed)
    lines_sb = [l.strip() for l in out_sb.splitlines() if l.strip()]
    step_q = lines_sb[0] if lines_sb else standalone_q

    # 3. Multi-Query leve (2 variantes) + Fusão RRF ponderada
    system_mq = (
        "Generate 2 different technical reformulations of the user's question for searching Ascend and CANN hardware manuals, "
        "using synonyms and technical variations. Write one per line, without numbering or explanations."
    )
    out_mq = llm([{"role": "system", "content": system_mq}, {"role": "user", "content": standalone_q}], tr, REWRITE_TEMP, seed)
    mq_variants = clean_lines(out_mq, standalone_q, 2)

    # Lista de buscas: peso 2x para standalone_q (repetida no ranking) + step_q + variantes
    all_queries = [standalone_q, step_q] + mq_variants
    rankings, pool = [], {}

    for idx, qq in enumerate(all_queries):
        hs = search(col, qq, tr)
        rank_ids = [h["id"] for h in hs]
        rankings.append(rank_ids)
        if idx == 0:  # Dá peso extra para a pergunta principal exata
            rankings.append(rank_ids)
        for h in hs:
            if h["id"] not in pool or h["distance"] < pool[h["id"]]["distance"]:
                pool[h["id"]] = h

    hits = []
    for cid, score in rrf(rankings)[:K]:
        h = dict(pool[cid])
        h["rrf"] = round(score, 5)
        hits.append(h)

    return standalone_q, hits

VARIANTS = {"baseline": r_baseline, "stepback": r_stepback, "multiquery": r_multiquery, "memory": r_memory, "combined": r_combined}

# Suíte de Avaliação Comparativa e LLM-as-a-Judge

Definição das perguntas de teste do domínio Huawei Ascend com gabaritos factuais verificados (`reference`), verificação por palavras-chave (`must_contain`) e avaliação automática imparcial via **LLM-as-a-Judge** (`qwen2.5:7b`).

In [62]:
os.makedirs("results", exist_ok=True)

def norm(s):
    import unicodedata
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s.replace("*", ""))

def Q(id, tier, question, must_contain, gold_docs, reference, expect_refusal=False, history=None):
    return {
        "id": id,
        "tier": tier,
        "question": question,
        "must_contain": must_contain,
        "gold_docs": gold_docs,
        "reference": reference,
        "expect_refusal": expect_refusal,
        "history": history or [],
    }

ATC_DOC = "ATCToolInstructions"
WP_DOC  = "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08"
ENV_DOC = "EnvironmentDeploymentGuide"
DEV_DOC = "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_"
TBE_DOC = "TBECustomOperatorDevelopmentGuide_CLI_"

QUESTIONS = [
    # ---------------- direct (exact parameters & specs) ----------------
    Q("q1", "direct", "What does the --input_fp16_nodes parameter do in the ATC tool?",
      ["fp16", "input_fp16_nodes"], [ATC_DOC],
      reference="The --input_fp16_nodes parameter specifies the name of the input node(s) with input data of type FP16 (for example, 'node_name1;node_name2'). It is mutually exclusive with --insert_op_conf."),

    Q("q2", "direct", "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
      ["it21dmda", "it21vdmb", "8", "11"], [WP_DOC],
      reference="According to Table 1-1 of the White Paper: 1) Power input: IT21DMDA accepts 5 V to 28 V, while IT21VDMB accepts 12 V. 2) MIPI-CSI port: IT21DMDA has a 22-pin port (without cables), while IT21VDMB has a 51-pin port (with cables). 3) Number of GPIOs: IT21DMDA has 8 GPIOs, while IT21VDMB has 11 GPIOs. Providing the GPIO difference (8 vs 11) and MIPI-CSI or power difference is sufficient."),

    Q("q3", "direct", "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?",
      ["512", "partition|space|sdcard|sd card"], [ENV_DOC],
      reference="The error occurs because an unallocated space of 512 MB is not reserved on the SD card of the Atlas 200 DK for Driver upgrade (often caused by using an older card creation script). It can be checked with 'fdisk -l' and fixed by resizing the partition (e.g. using GParted on the Ubuntu server to leave 512 MiB of unallocated space)."),

    Q("q4", "direct", "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?",
      ["16", "c0|nc1hwc0"], [ATC_DOC, TBE_DOC, DEV_DOC],
      reference="In the 5D tensor format NC1HWC0, N is batch size, H is height, W is width, C0 is the size of the cube / matrix computing unit in the AI Core, and C1 = (C + C0 - 1) / C0. For FP16, C0 is 16 (and for INT8, C0 is 32)."),

    # ---------------- multi: multiple parameters or tables ----------------
    Q("q5", "multi", "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
      ["matrix_r0c0", "256|298"], [ATC_DOC, DEV_DOC],
      reference="In Table 4-1 (YUV420SP_U8 to RGB / RGB888_U8), for JPEG images the CSC matrix coefficients are matrix_r0c0: 256, matrix_r0c1: 0, matrix_r0c2: 359, matrix_r1c0: 256, matrix_r1c1: -88, matrix_r1c2: -183, matrix_r2c0: 256, matrix_r2c1: 454, matrix_r2c2: 0 (with input_bias 0, 128, 128). For BT.601 Narrow, the coefficients are 298, 0, 409, 298, -100, -208, 298, 516, 0. Citing either the JPEG column (256...) or the BT.601 column (298...) from the YUV420SP_U8 to RGB table is acceptable."),

    Q("q6", "multi", "What is the typical power consumption and power input voltage range of the Atlas 200 DK?",
      ["12", "20|36|28"], [WP_DOC],
      reference="According to the White Paper (Sections 3.1 and 4.4): Typical power consumption is 20 W, and the external power supply adapter must be >= 36 W (12 V 3 A configured by default). The power input voltage is 12 V if the PCB is IT21VDMB, and 5 V to 28 V if the PCB is IT21DMDA. Mentioning the 12V / 5V-28V input voltages and either the 20W typical power or >=36W power supply requirement is correct."),

    # ---------------- paraphrase: vocabulary gap ----------------
    Q("q7", "paraphrase", "How do I configure channel mean subtraction and variance normalization in the AIPP configuration file?",
      ["mean_chn|var_reci_chn|min_chn"], [ATC_DOC, DEV_DOC],
      reference="In the AIPP configuration file (aipp_op), channel normalization is configured using mean_chn_0..3 (mean value of each channel), min_chn_0..3 (minimum value of each channel), and var_reci_chn_0..3 (variance / reciprocal factor of each channel), following the rule: pixel_out = (pixel_in - mean_chn - min_chn) * var_reci_chn."),

    Q("q8", "paraphrase", "What should I do if the Atlas board cannot establish connection to the Ubuntu server via USB?",
      ["usb", "ip|nic|cable|led|sd"], [ENV_DOC, "UserGuide2021", "UserGuide2020"],
      reference="According to the troubleshooting guide (What Do I Do If the Atlas 200 DK Cannot Connect to the Ubuntu Server?): 1) Check SD card preparation logs and ensure LED1/LED2 indicators are normal. 2) Check the USB cable and verify if the virtual USB NIC exists and its IP address (e.g. 192.168.1.x) is configured on the Ubuntu server. 3) Alternatively, try connecting via network cable (NIC mode) or connect via a serial cable (UART) using a debugging tool (e.g. IPOP / PL2303 driver) to inspect boot logs."),

    # ---------------- followup: depends on conversation history ----------------
    Q("q9", "followup", "And what is its value for INT8 data types?",
      ["32"], [ATC_DOC, TBE_DOC, DEV_DOC],
      reference="For INT8 data types in the NC1HWC0 5D format, C0 is 32.",
      history=[{"role": "user", "content": "What is the value of C0 for FP16 in the NC1HWC0 5D tensor format?"},
               {"role": "assistant", "content": "In the NC1HWC0 5D tensor format, C0 represents the size of the cube unit in the AI Core and equals 16 for FP16."}]),

    # ---------------- none: out of domain refusal ----------------
    Q("q10", "none", "What is the GPU clock speed of Nvidia RTX 4090?",
      [], [],
      reference="Out of domain question. The answer must state that the information is not provided in the context.",
      expect_refusal=True),
    # ---------------- direct (exact parameters & specs) ----------------
    Q("q11", "direct", "Why does DMalloc return error code 16847020 (HIAI_GRAPH_MEMORY_POOL_STOPPED) in a single-process multi-graph scenario?",
      ["first graph", "memory pool"], ["ApplicationDevelopmentGuide_MindStudio_"],
      reference="Graphs of the same process on the host share one memory pool maintained by the first graph. When the first graph is destroyed, the memory pool is stopped, causing DMalloc to return 16847020 (HIAI_GRAPH_MEMORY_POOL_STOPPED) until the first graph is completely destroyed and the second graph becomes the first graph. It can be avoided by not destroying the first graph or by adding a retry mechanism."),

    Q("q12", "direct", "What are the default memory allocations for weights and feature maps during TensorFlow training on Ascend, and which parameters adjust them?",
      ["5", "26", "graph_memory_max_size|variable_memory_max_size"], ["TensorFlowNetworkModelPortingandTrainingGuide"],
      reference="By default, 5 GB is allocated for weight memory and 26 GB is allocated for feature map memory (with a combined limit of 31 GB). They can be adjusted using variable_memory_max_size and graph_memory_max_size in NPURunConfig or session configuration."),

    Q("q13", "direct", "In the TBE operator description JSON file inside kernel_meta, what do the magic values RT_DEV_BINARY_MAGIC_ELF and RT_DEV_BINARY_MAGIC_ELF_AICPU indicate?",
      ["ai core", "ai cpu"], [TBE_DOC, DEV_DOC],
      reference="In the kernel_meta JSON file, 'RT_DEV_BINARY_MAGIC_ELF' indicates that the operator running target is the AI Core (with a .o binary file suffix), whereas 'RT_DEV_BINARY_MAGIC_ELF_AICPU' indicates that the running target is the AI CPU (with a .so binary file suffix)."),

    Q("q14", "direct", "What are the default port numbers and initial login usernames for the Mind Studio GUI and the Profiling tool?",
      ["8888", "8099", "mindstudioadmin", "msvpadmin"], ["MindStudioInstallationGuideUbuntuX86"],
      reference="For the Mind Studio GUI, the default port is 8888 and the default username is MindStudioAdmin (initial password: Huawei123@). For the Profiling GUI, the default port is 8099 and the default username is msvpadmin (initial password: Admin12#$)."),

    # ---------------- multi: multiple parameters or tables ----------------
    Q("q15", "multi", "How is the required output memory size calculated for a 300 x 300 YUV420SP_UV image in DVPP, and which Matrix APIs must be used to allocate and free this memory?",
      ["304", "1.5|3/2", "hiai_dvpp_dmalloc", "hiai_dvpp_dfree"], ["ApplicationDevelopmentGuide_MindStudio_", DEV_DOC],
      reference="Due to DVPP alignment rules (width aligned to 16 pixels and height to 2 pixels), a 300 x 300 YUV420SP_UV image is rounded up to 304 x 300. Because each YUV420SP pixel requires 1.5 (3/2) bytes, the output memory size is 304 x 300 x 3/2 bytes. This memory must be allocated and freed in pairs on the device side using HIAI_DVPP_DMalloc (or HIAIMemory::HIAI_DVPP_DMalloc) and HIAI_DVPP_DFree (or HIAIMemory::HIAI_DVPP_DFree)."),

    Q("q16", "multi", "Which pins on the Atlas 200 DK must be connected with a jumper cap to prepare an SD card without a card reader, and what are the default IP addresses configured in make_sd_card.py for the USB NIC and network port?",
      ["16", "18", "192.168.1.2", "192.168.0.2"], [ENV_DOC, "UserGuide2020", "UserGuide2021"],
      reference="To prepare an SD card without a card reader, place a jumper cap or jumper wire over pin 16 and pin 18 on the Atlas 200 DK while powered off. In make_sd_card.py, the default IP address for the USB virtual NIC (USB_CARD_DEFAULT_IP) is 192.168.1.2, and the default IP address for the physical NIC (NETWORK_CARD_DEFAULT_IP) is 192.168.0.2."),

    # ---------------- paraphrase: vocabulary gap ----------------
    Q("q17", "paraphrase", "How should VDEC engines and thread counts be configured in the graph file to decode multiple video streams concurrently without frame order issues?",
      ["thread_num", "1", "16|one engine|multiple vdec"], ["ApplicationDevelopmentGuide_MindStudio_"],
      reference="In the graph configuration file, configure multiple VDEC engines so that each video stream fixedly corresponds to one VDEC engine (up to 16 video streams per graph), and set thread_num to 1 in each VDEC engine segment so that one engine corresponds to one thread."),

    Q("q18", "paraphrase", "How can I speed up slow offline model conversion when running on an Arm (AArch64) server?",
      ["numactl", "16-31|localalloc|te_parallel_compiler"], [ATC_DOC, DEV_DOC],
      reference="On an Arm (AArch64) development server, you can accelerate offline model conversion by installing numactl and binding CPU cores 16-31 using 'numactl -C 16-31 --localalloc <command>' (with atc or omg). In ATC, you can also enable parallel operator compilation via the TE_PARALLEL_COMPILER environment variable."),

    # ---------------- followup: depends on conversation history ----------------
    Q("q19", "followup", "And what value indicates that it is a VD version?",
      ["004"], [ENV_DOC, "UserGuide2020", "UserGuide2021"],
      reference="When checking '/proc/cmdline' on the Atlas 200 DK, a value of boardid=004 indicates that the Atlas 200 AI accelerator module is a VD version.",
      history=[{"role": "user", "content": "How can I check if the Atlas 200 AI accelerator module is a VC or earlier version using the command line?"},
               {"role": "assistant", "content": "You can run 'cat /proc/cmdline' on the Atlas 200 DK and inspect the boardid parameter. A value of boardid=000 indicates a VC or earlier version."}]),

    # ---------------- none: out of domain refusal ----------------
    Q("q20", "none", "How many Tensor Cores does the Nvidia H100 Hopper GPU have?",
      [], [],
      reference="Out of domain question. The answer must state that the information is not provided in the context.",
      expect_refusal=True),
]

QUESTIONS.sort(key=lambda q: int(q["id"][1:]))

with open("results/questions.json", "w", encoding="utf-8") as f:
    json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)
if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
    (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
    with open(Path(BASE_DIR) / "results" / "questions.json", "w", encoding="utf-8") as f:
        json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)

print(f"[OK] Benchmark suite: {len(QUESTIONS)} questions configured with verified Huawei manual references.")

[OK] Benchmark suite: 20 questions configured with verified Huawei manual references.


In [63]:
REFUSAL_RE = re.compile(
    r"do not have (enough|sufficient) information|don't have (enough|sufficient) information"
    r"|insufficient information|not enough information|cannot (answer|determine|find)"
    r"|(context|documents?|corpus|manuals?) does not (contain|mention|provide|specify)"
    r"|does not (contain|mention|provide|specify) (any |enough |sufficient )?(information|details)"
    r"|not (mentioned|provided|specified|found|available|covered|addressed) in the (provided |given |attached )?(context|documents?|corpus|manuals?)"
    r"|no information (available|provided|found)",
    re.I,
)

def has_kw(text_norm, kw):
    return any(norm(alt).strip() in text_norm for alt in kw.split("|"))

def grade(answer, q):
    """
    Evaluate answer using LLM-as-a-judge (qwen2.5:7b) comparing candidate answer
    against factual ground-truth reference, with deterministic refusal handling.
    """
    a = norm(answer)
    refused = bool(REFUSAL_RE.search(a))
    kw_ok = refused if q["expect_refusal"] else all(has_kw(a, kw) for kw in q["must_contain"])

    # 1. Deterministic handling for out-of-domain refusal questions (e.g. q10, q20)
    if q["expect_refusal"]:
        if refused:
            grade.last = {"reason": "Correctly refused out-of-domain question based on context."}
            return True, True
        else:
            grade.last = {"reason": "Failed to refuse out-of-domain question."}
            return False, False

    # 2. If it is an answerable question but the RAG model refused to answer
    if refused:
        grade.last = {"reason": "Model stated insufficient information instead of answering."}
        return False, True

    reference = q.get("reference") or "; ".join(q["must_contain"])
    hist = "\n".join(f"{t['role']}: {t['content']}" for t in q.get("history", [])) or "(no history)"

    judge_prompt = (
        f"Conversation history:\n{hist}\n\n"
        f"Question: {q['question']}\n\n"
        f"Ground truth reference:\n{reference}\n\n"
        f"Candidate answer to evaluate:\n{answer}\n\n"
        "Evaluate whether the candidate answer is CORRECT based on the ground truth reference:\n"
        "1. It must accurately answer the question and align with the key facts in the reference.\n"
        "2. Do NOT penalize the candidate answer for including additional valid technical steps or details from the manual that are not in the short reference summary.\n"
        "3. Mark correct=false ONLY if the answer misses the core technical answer or directly contradicts the reference numbers/facts.\n"
        'Reply ONLY in JSON format: {"reason": "a short sentence", "correct": true or false}'
    )

    try:
        r = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": "You are a fair, technically accurate evaluator of RAG system answers."},
                {"role": "user", "content": judge_prompt},
            ],
            temperature=0, seed=0, response_format={"type": "json_object"}
        )
        v = json.loads(r.choices[0].message.content)
        val = v.get("correct") if "correct" in v else v.get("correto")
        judge_ok = val is True or str(val).strip().lower() in ("true", "1")
        reason = str(v.get("reason") or v.get("motivo") or "")[:180]
    except Exception as e:
        judge_ok, reason = kw_ok, f"judge fallback ({type(e).__name__}): keywords used"

    grade.last = {"reason": reason}
    return judge_ok, refused

def run_variant(label, fn, questions, run, col):
    path = f"results/{label}_r{run}.csv"
    if os.path.exists(path) and not FORCE:
        return pd.read_csv(path)
    if "BASE_DIR" in globals() and not FORCE:
        drive_path = Path(BASE_DIR) / "results" / f"{label}_r{run}.csv"
        if drive_path.exists():
            return pd.read_csv(drive_path)
    rows = []
    for qi, q in enumerate(questions):
        tr, seed = Trace(), SEED_BASE + 1000 * run + qi
        t0 = time.time()
        q_used, hits = fn(q, col, tr, seed)
        answer = generate_answer(q_used, hits, tr, seed)
        latency = time.time() - t0

        correct, refused = grade(answer, q)
        judge_reason = grade.last.get("reason", "")

        docs = {h["doc"] for h in hits}
        recall = (sum(d in docs for d in q["gold_docs"]) / len(q["gold_docs"])) if q["gold_docs"] else np.nan
        ctx = norm(format_context(hits))
        ctx_keys = all(has_kw(ctx, kw) for kw in q["must_contain"]) if q["must_contain"] else np.nan

        rows.append({
            "judge_reason": judge_reason,
            "label": label, "run": run, "id": q["id"], "tier": q["tier"], "question": q["question"],
            "question_used": q_used, "queries": json.dumps(tr.queries, ensure_ascii=False),
            "retrieved": " | ".join(f"{h['id']}({h['distance']:.3f})" for h in hits),
            "gold_docs": ";".join(q["gold_docs"]), "gold_recall": recall, "ctx_has_keys": ctx_keys,
            "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": tr.llm_calls, "embed_calls": tr.embed_calls, "latency_s": round(latency, 2),
        })
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
        (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
        df.to_csv(Path(BASE_DIR) / "results" / f"{label}_r{run}.csv", index=False)
    return df

In [64]:
# Evaluates variants on questions.json (optional quick run or batch evaluation)
RUN_FULL_EVALUATION = True  # Set to True when ready to execute all variants in Google Colab

if RUN_FULL_EVALUATION:
    with open("results/questions.json", encoding="utf-8") as f:
        questions = json.load(f)

    # Warmup call
    llm([{"role": "user", "content": "ok"}], Trace(), 0.0, 0)
    embed(["ok"], prefix="search_query: ")

    for label, fn in VARIANTS.items():
        for run in range(1, N_RUNS + 1):
            t0 = time.time()
            df = run_variant(label, fn, questions, run, col)
            print(f"{label:<11} r{run}  acc={df['correct'].mean():.0%}  llm_calls={df['llm_calls'].mean():.1f}  ({time.time()-t0:.0f}s)")
        combined_df = pd.concat([pd.read_csv(f"results/{label}_r{r}.csv") for r in range(1, N_RUNS + 1)])
        combined_df.to_csv(f"results/results_{label}.csv", index=False)
        if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
            (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
            combined_df.to_csv(Path(BASE_DIR) / "results" / f"results_{label}.csv", index=False)

    try:
        ollama_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json().get("version")
    except Exception:
        ollama_version = None
    run_cfg = {
        "gen_model": GEN_MODEL, "judge_model": JUDGE_MODEL, "emb_model": EMB_MODEL,
        "ollama_version": ollama_version, "k": K, "n_runs": N_RUNS, "seed_base": SEED_BASE,
        "gen_temp": GEN_TEMP, "rewrite_temp": REWRITE_TEMP, "n_variants": N_VARIANTS, "rrf_c": RRF_C,
        "memory_distractor": MEMORY_DISTRACTOR, "n_chunks": len(rows), "n_questions": len(questions),
        "date": time.strftime("%Y-%m-%d %H:%M"),
    }
    with open("results/run_config.json", "w", encoding="utf-8") as f:
        json.dump(run_cfg, f, indent=2, ensure_ascii=False)
    if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
        (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
        with open(Path(BASE_DIR) / "results" / "run_config.json", "w", encoding="utf-8") as f:
            json.dump(run_cfg, f, indent=2, ensure_ascii=False)
    print("\n[OK] Comparative results exported to results/")
else:
    print("[INFO] Set RUN_FULL_EVALUATION = True to execute the complete variant benchmark with LLM-as-a-judge.")

baseline    r1  acc=80%  llm_calls=1.0  (54s)
stepback    r1  acc=70%  llm_calls=2.0  (49s)
multiquery  r1  acc=75%  llm_calls=2.0  (58s)
memory      r1  acc=75%  llm_calls=2.0  (51s)
combined    r1  acc=90%  llm_calls=3.1  (57s)

[OK] Comparative results exported to results/


# Comparativo de Desempenho e Motivo da Avaliação do Juiz

Exibição detalhada das respostas de cada variante com o veredito (`[OK]` / `[FAIL]`) e a justificativa emitida pelo modelo juiz (`qwen2.5:7b`), acompanhada do resumo de acurácia comparativa.

In [65]:
from IPython.display import display
pd.set_option("display.max_colwidth", 300)

summary_rows = []
for label in VARIANTS:
    csv_path = f"results/results_{label}.csv"
    if not os.path.exists(csv_path):
        continue
    d = pd.read_csv(csv_path)

    t = d[["id", "question", "answer"]].copy()
    t["result"] = d["correct"].map({True: "[OK]", False: "[FAIL]"})
    t["judge_reason"] = d["judge_reason"]

    correct_cnt, total_cnt = int(d["correct"].sum()), len(d)
    summary_rows.append({
        "variant": label,
        "correct": correct_cnt,
        "total": total_cnt,
        "accuracy": f"{correct_cnt / total_cnt:.0%}",
    })

    print(f"\n===== Variant: {label} =====")
    display(t.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    print(f"Accuracy: {correct_cnt}/{total_cnt} = {correct_cnt / total_cnt:.0%}")

if summary_rows:
    print("\n===== SUMMARY COMPARISON =====")
    display(pd.DataFrame(summary_rows).style.hide(axis="index"))


===== Variant: baseline =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides data in FP16 format, and this node needs to be handled specially during the conversion process. Specifically: - You should enclose the name of each node in double quotation marks ("""") and separate the nodes by semicolons (;). - For example, the command might look like this: `atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data""` This parameter is particularly useful when dealing with models that have specific input nodes providing FP16 data, ensuring that these nodes are correctly processed and that the input data format is maintained during the conversion to the `.om` model format.",[OK],accurately describes the purpose and usage of the --input_fp16_nodes parameter
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,"The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port, but without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port, and the port comes with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences reflect improvements and enhancements in the newer IT21VDMB version of the Atlas 200 DK, particularly in terms of power requirements and connectivity options.",[OK],accurately summarizes the main hardware differences
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. Here are the steps to address this issue: 1. **Check the Current Partition Space:** - Connect the SD card to the Ubuntu server over the USB port. - Run the `fdisk -l` command as the root user to check the current partition space. ```sh fdisk -l ``` - If the output shows: - Available sectors: 62,333,952 - Allocated sectors: 61,285,375 - Unallocated space: 512 MB (Calculated as: (62333952 - 61285375) x 512/1024/1024) - This indicates sufficient unallocated space. - Or, if the output shows: - Available sectors: 6,233,952 - Allocated sectors: 6,233,951 - This indicates insufficient unallocated space. 2. **Resize the Partition:** - If the unallocated space is insufficient, you need to resize the partition to create the required unallocated space. - Use a card reader to insert the SD card and connect it to the Ubuntu server over the USB port. - Alternatively, if a card reader is not available, place a jumper cap or wire over pins 16 and 18 on the Atlas 200 DK (power off the device first). - Run the following commands to install and start the GParted tool: ```sh apt-get install gparted gparted ``` - Switch to the corresponding device (SD card). - Unmount the partition: - Right-click /dev/sdb3 and choose Unmount. - Resize the partition: - Ensure the Unused space of the last partition is greater than 512 MB. - If the Unused space is smaller, delete unnecessary files from the /home/HwHiAiUser directory on the A

Accuracy: 16/20 = 80%

===== Variant: stepback =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],accurately describes the parameter and its usage
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],accurately summarizes the main hardware differences
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],accurately addresses the cause and resolution
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - N: Batch size - C1: Number of feature maps, which is derived from the original C dimension by dividing it by C0: \( C1 = \left\lceil \frac{C}{C0} \right\rceil \) - H: Height of the feature map - W: Width of the feature map - C0: The size of the matrix computing unit in the AI Core, which is equal to 16 for the FP16 type For the FP16 data type, C0 is 16. This means that the C dimension is divided into segments of 16 elements each, with the last segment possibly being padded with zeros if the division is not exact to ensure that the segments are of equal size.",[OK],"The answer correctly identifies the meaning of C0 for FP16 and the structure of the NC1HWC0 format, though it uses C1 instead of C0 for the derived dimension."
q5,Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8

Accuracy: 14/20 = 70%

===== Variant: multiquery =====


...

id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],accurately describes the parameter and its usage
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],accurately summarizes the main hardware differences
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],accurately addresses the cause and resolution
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - N: Batch size - C1: Number of feature maps, which is derived from the original C dimension by dividing it by C0: \( C1 = \left\lceil \frac{C}{C0} \right\rceil \) - H: Height of the feature map - W: Width of the feature map - C0: The size of the matrix computing unit in the AI Core, which is equal to 16 for the FP16 type For the FP16 data type, C0 is 16. This means that the C dimension is divided into segments of 16 elements each, with the last segment possibly being padded with zeros if the division is not exact to ensure that the segments are of equal size.",[OK],"The answer correctly identifies the meaning of C0 for FP16 and the structure of the NC1HWC0 format, though it uses C1 instead of C0 for the derived dimension."
q5,Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8

Accuracy: 15/20 = 75%

===== Variant: combined =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],accurately describes the parameter and its usage
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],accurately summarizes the main hardware differences
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],accurately addresses the cause and resolution
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - **N**: Batch size - **C1**: Number of 16-element channels derived from the original C dimension - **H**: Height - **W**: Width - **C0**: Size of the matrix computing unit in AI Core, which is equal to 16 for the FP16 data type. This means that the C dimension is divided by 16, and any remainder is padded with zeros to align with the C0 size. In summary, for FP16 data type, C0 is 16, and C1 is calculated as \((C + 15) // 16\), ensuring that the data is stored consecutively in memory.",[OK],The answer correctly identifies C0 as 16 for FP16 and provides a clear explanation of the C1 calculation.
q5,Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?,"For JPEG images, the CSC matrix coefficients used in AIPP to convert YUV420SP_U8 to RGB888_U8 are as

Accuracy: 18/20 = 90%

===== SUMMARY COMPARISON =====


variant,correct,total,accuracy
baseline,16,20,80%
stepback,14,20,70%
multiquery,15,20,75%
memory,15,20,75%
combined,18,20,90%


### Avaliando o Juiz
#### Nesta célula é avaliado o parecer do juiz com base em respostas retornadas pelo `Gemini Notebook`

In [66]:
import os, re, json, html
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_colwidth", 300)
os.makedirs("results", exist_ok=True)

# Anchor refusal patterns to context/documents/corpus/manuals or information/details
REFUSAL_RE = re.compile(
    r"do not have (enough|sufficient) information|don't have (enough|sufficient) information"
    r"|insufficient information|not enough information|cannot (answer|determine|find)"
    r"|(context|documents?|corpus|manuals?) does not (contain|mention|provide|specify)"
    r"|does not (contain|mention|provide|specify) (any |enough |sufficient )?(information|details)"
    r"|not (mentioned|provided|specified|found|available|covered|addressed) in the (provided |given |attached )?(context|documents?|corpus|manuals?)"
    r"|no information (available|provided|found)",
    re.I,
)

def resolve_gemini_md_path() -> Path:
    rel_target = Path("data") / "raw" / "gemini_notebook" / "gemini_notebook_answers.md"
    candidates = [rel_target]
    if "BASE_DIR" in globals():
        candidates.append(Path(BASE_DIR) / rel_target)
    if "ALT_ROOTS" in globals():
        for root in ALT_ROOTS:
            candidates.append(Path(root) / rel_target)
            candidates.append(Path(root) / "gemini_notebook_answers.md")
    for cand in candidates:
        if cand.exists() and cand.is_file():
            return cand
    if "_download_file" in globals():
        rel_target.parent.mkdir(parents=True, exist_ok=True)
        if _download_file(str(rel_target).replace("\\", "/"), rel_target):
            return rel_target
    raise FileNotFoundError(f"Could not locate {rel_target} in workspace or Google Drive.")

def load_gemini_notebook_answers(md_path: Path = None) -> dict:
    if md_path is None:
        md_path = resolve_gemini_md_path()
    md_text = md_path.read_text(encoding="utf-8")
    header_re = re.compile(r"^###\s+(q\d+)\s*\(([^)]+)\)\s*$", re.MULTILINE)
    matches = list(header_re.finditer(md_text))
    parsed = {}
    for idx, m in enumerate(matches):
        qid = m.group(1).strip()
        tier = m.group(2).strip().lower()
        start = m.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(md_text)
        block = md_text[start:end].strip()

        history = []
        user_m = re.search(r"\*\s*\*\*User:\*\*\s*(.+)", block)
        asst_m = re.search(r"\*\s*\*\*Assistant:\*\*\s*(.+)", block)
        if user_m and asst_m:
            history = [
                {"role": "user", "content": user_m.group(1).strip()},
                {"role": "assistant", "content": asst_m.group(1).strip()},
            ]

        q_m = re.search(r"\*\s*\*\*Question:\*\*\s*(.+)", block)
        ans_m = re.search(
            r"\*\s*\*\*Answer:\*\*\s*(.*?)(?=\n\*\s*\*\*Source Document Titles?:|\Z)",
            block,
            flags=re.DOTALL,
        )
        src_m = re.search(r"\*\s*\*\*Source Document Titles?:\*\*\s*(.+)", block)

        parsed[qid] = {
            "id": qid,
            "tier": tier,
            "question": q_m.group(1).strip() if q_m else "",
            "answer": ans_m.group(1).strip() if ans_m else "",
            "source_docs": src_m.group(1).strip() if src_m else "",
            "history": history,
            "expect_refusal": (tier == "none"),
        }
    return parsed

def _load_eval_questions(gemini_answers: dict) -> list:
    base_questions = []
    if "QUESTIONS" in globals() and isinstance(QUESTIONS, list):
        base_questions = [dict(q) for q in QUESTIONS]
    q_candidates = [Path("results/questions.json")]
    if "BASE_DIR" in globals():
        q_candidates.append(Path(BASE_DIR) / "results" / "questions.json")
    if "ALT_ROOTS" in globals():
        for root in ALT_ROOTS:
            q_candidates.append(Path(root) / "results" / "questions.json")
    for qc in q_candidates:
        if qc.exists() and qc.is_file():
            try:
                loaded = json.loads(qc.read_text(encoding="utf-8"))
                if len(loaded) > len(base_questions):
                    base_questions = loaded
            except Exception:
                pass

    by_id = {q["id"]: dict(q) for q in base_questions}
    for qid, md_item in gemini_answers.items():
        if qid not in by_id:
            by_id[qid] = {
                "id": qid,
                "tier": md_item["tier"],
                "question": md_item["question"],
                "must_contain": [],
                "gold_docs": [],
                "reference": md_item["answer"],
                "expect_refusal": md_item["expect_refusal"],
                "history": md_item.get("history", []),
            }
    return sorted(by_id.values(), key=lambda q: int(re.sub(r"\D", "", q["id"]) or 0))

GEMINI_MD_PATH = resolve_gemini_md_path()
GEMINI_ANSWERS = load_gemini_notebook_answers(GEMINI_MD_PATH)
print(f"[OK] Loaded {len(GEMINI_ANSWERS)} ground-truth answers from {GEMINI_MD_PATH}")

eval_questions = _load_eval_questions(GEMINI_ANSWERS)
judge_eval_rows = []

for q in eval_questions:
    qid = q["id"]
    if qid not in GEMINI_ANSWERS:
        continue
    md_item = GEMINI_ANSWERS[qid]
    gemini_answer = md_item["answer"]

    # Evaluate the Gemini Notebook ground-truth answer using the default LLM-as-a-Judge
    judge_ok, refused = grade(gemini_answer, q)
    judge_reason = getattr(grade, "last", {}).get("reason", "")

    judge_eval_rows.append({
        "id": qid,
        "tier": q["tier"],
        "question": q["question"],
        "gemini_answer": gemini_answer,
        "reference": q.get("reference", ""),
        "correct": judge_ok,
        "refused": refused,
        "result": "[OK]" if judge_ok else "[FAIL]",
        "judge_reason": judge_reason,
    })

judge_eval_df = pd.DataFrame(judge_eval_rows)
judge_eval_df.to_csv("results/judge_eval_gemini_notebook.csv", index=False)
if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
    (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
    judge_eval_df.to_csv(Path(BASE_DIR) / "results" / "judge_eval_gemini_notebook.csv", index=False)

display_cols = ["id", "tier", "question", "gemini_answer", "result", "judge_reason"]
print("\n===== LLM-as-a-Judge Evaluation on Gemini Notebook Answers =====")
display(
    judge_eval_df[display_cols]
    .style.hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
)

correct_cnt = int(judge_eval_df["correct"].sum())
total_cnt = len(judge_eval_df)
print(f"Judge Accuracy (Agreement with Gemini Notebook): {correct_cnt}/{total_cnt} = {correct_cnt / total_cnt:.0%}")

misjudged = judge_eval_df[~judge_eval_df["correct"]]
if not misjudged.empty:
    print(f"\n[WARNING] Judge disagreed on {len(misjudged)} question(s):")
    for _, row in misjudged.iterrows():
        print(f"  - {row['id']} ({row['tier']}): {row['judge_reason']}")

[OK] Loaded 20 ground-truth answers from /content/drive/MyDrive/RAG_Atlas200DK/data/raw/gemini_notebook/gemini_notebook_answers.md

===== LLM-as-a-Judge Evaluation on Gemini Notebook Answers =====


id,tier,question,gemini_answer,result,judge_reason
q1,direct,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter specifies the name of the input node(s) with input data of type FP16 (for example, `""node_name1;node_name2""`, where each node is enclosed in double quotation marks and separated by semicolons). This parameter is optional (default value: `N/A`), is mutually exclusive with `--insert_op_conf`, and can be used in conjunction with `--is_input_adjust_hw_layout` (which, when set to `true`, allows `input_fp16_nodes` to accept `NC1HWC0` tensors of type FP16).",[OK],The answer provides the correct information about the --input_fp16_nodes parameter and its mutual exclusivity with --insert_op_conf.
q2,direct,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,"According to Table 1-1 (*PCB specification comparisons*), Section 4.4, and Section 4.6.4 of the white paper, the main hardware differences between `IT21DMDA` (Old PCB) and `IT21VDMB` (New PCB) are: * **Power input:** `IT21DMDA` accepts **5 V to 28 V**, whereas `IT21VDMB` accepts **12 V**. * **MIPI-CSI port:** `IT21DMDA` has a **22-pin** port without cables, whereas `IT21VDMB` has a **51-pin** port with cables. * **Number of GPIOs:** `IT21DMDA` provides **8** GPIO pins (pins 26, 27, and 28 are NC pins), whereas `IT21VDMB` provides **11** independent GPIO pins.",[OK],Accurate and aligned with the reference.
q3,direct,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"This error occurs during the execution of `./minirc_install_phase1.sh` because an unallocated space of **512 MB** is not reserved on the SD card of the Atlas 200 DK for the Driver upgrade, which may be caused by using an older card creation script. You can verify the unallocated space as the `root` user by running `fdisk -l` to compare total available sectors against allocated sectors, and resolve the issue by connecting the SD card to the Ubuntu server over USB and resizing the partition (for example, using GParted) to reserve the required 512 MB of unallocated space.",[OK],"The candidate answer aligns with the key facts provided in the reference, including the cause and resolution of the error."
q4,direct,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"To improve data access efficiency in the Ascend AI Software Stack, tensor data is stored in the Huawei-defined 5D format `NC1HWC0`: * **Dimensions:** `N` represents the batch size, `H` is the height, `W` is the width, `C0` is the size of the cube / matrix computing unit in the AI Core (closely related to the microarchitecture), and `C1 = (C + C0 - 1) / C0` (if the division is not exact, the last data segment is padded with zeros to align with `C0`). * **Value of C0:** For **FP16**, `C0` is **16** (and for **INT8**, `C0` is **32**), and `C0` elements must be stored contiguously in memory. * **Conversion from NHWC:** The `NHWC` data is split along the `C` dimension into `C1` pieces of `NHWC0`, which are then arranged contiguously in memory to form `NC1HWC0`.",[OK],accurately describes the NC1HWC0 format and correctly states C0 for FP16
q5,multi,Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?,"In the AIPP configuration (`aipp_op` with `input_format : YUV420SP_U8`, `csc_switch : true`, and `rbuv_swap_switch : false`), the Color Space Conversion (CSC) matrix coefficients are: * **For JPEG images** (with `input_bias_0 : 0`, `input_bias_1 : 128`, `input_bias_2 : 128`): `matrix_r0c0 : 256`, `matrix_r0c1 : 0`, `matrix_r0c2 : 359`, `matrix_r1c0 : 256`, `matrix_r1c1 : -88`, `matrix_r1c2 : -183`, `matrix_r2c0 : 256`, `matrix_r2c1 : 454`, `matrix_r2c2 : 0`. * **For BT.601 Narrow** (with `input_bias_0 : 16`, `input_bias_1 : 128`, `input_bias_2 : 128`): `matrix_r0c0 : 298`, `matrix_r0c1 : 0`, `matrix_r0c2 : 409`, `matrix_r1c0 : 29

Judge Accuracy (Agreement with Gemini Notebook): 20/20 = 100%


### Parecer de Outro Juiz
#### Nesta célula é feita a mesma avaliação do RAG, porém com base nas respostas do `Gemini Notebook`

In [67]:
if "GEMINI_ANSWERS" not in globals():
    GEMINI_ANSWERS = load_gemini_notebook_answers()

eval_questions = (
    _load_eval_questions(GEMINI_ANSWERS)
    if "_load_eval_questions" in globals()
    else (QUESTIONS if "QUESTIONS" in globals() else json.load(open("results/questions.json", encoding="utf-8")))
)

# Build question suite grounded on the Gemini Notebook Markdown answers
QUESTIONS_GEMINI_MD = []
for q in eval_questions:
    qid = q["id"]
    q_md = dict(q)
    if qid in GEMINI_ANSWERS:
        md_item = GEMINI_ANSWERS[qid]
        q_md["reference"] = md_item["answer"]
        if not q_md.get("history") and md_item.get("history"):
            q_md["history"] = md_item["history"]
    QUESTIONS_GEMINI_MD.append(q_md)

QUESTIONS_GEMINI_MAP = {q["id"]: q for q in QUESTIONS_GEMINI_MD}

def grade_with_gemini_md(answer, q_md):
    """
    Evaluate RAG candidate answer using the Gemini Notebook Markdown answer
    as the ground-truth reference.
    """
    a = norm(answer)
    refused = bool(REFUSAL_RE.search(a))
    kw_ok = refused if q_md["expect_refusal"] else all(has_kw(a, kw) for kw in q_md["must_contain"])

    # 1. Deterministic handling for out-of-domain refusal questions (e.g. q10, q20)
    if q_md["expect_refusal"]:
        if refused:
            grade_with_gemini_md.last = {"reason": "Correctly refused out-of-domain question based on context."}
            return True, True
        else:
            grade_with_gemini_md.last = {"reason": "Failed to refuse out-of-domain question."}
            return False, False

    # 2. If it is an answerable question but the RAG model refused to answer
    if refused:
        grade_with_gemini_md.last = {"reason": "Model stated insufficient information instead of answering."}
        return False, True

    gemini_reference = q_md["reference"]
    hist = "\n".join(f"{t['role']}: {t['content']}" for t in q_md.get("history", [])) or "(no history)"

    judge_prompt = (
        f"Conversation history:\n{hist}\n\n"
        f"Question: {q_md['question']}\n\n"
        f"Gemini Notebook ground-truth answer:\n{gemini_reference}\n\n"
        f"Candidate RAG answer to evaluate:\n{answer}\n\n"
        "Evaluate whether the candidate RAG answer is CORRECT based on the Gemini Notebook ground-truth answer:\n"
        "1. It must accurately answer the question and align with the core technical facts in the Gemini Notebook answer.\n"
        "2. Do NOT penalize the candidate answer for omitting optional secondary details (such as file suffixes or alternative modes) or for including additional valid steps from the manual.\n"
        "3. Mark correct=false ONLY if the answer misses the core technical answer or directly contradicts the Gemini Notebook facts.\n"
        'Reply ONLY in JSON format: {"reason": "a short sentence", "correct": true or false}'
    )

    try:
        r = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": "You are a fair, technically accurate evaluator of RAG system answers using Gemini Notebook answers as ground truth."},
                {"role": "user", "content": judge_prompt},
            ],
            temperature=0, seed=0, response_format={"type": "json_object"}
        )
        v = json.loads(r.choices[0].message.content)
        val = v.get("correct") if "correct" in v else v.get("correto")
        judge_ok = val is True or str(val).strip().lower() in ("true", "1")
        reason = str(v.get("reason") or v.get("motivo") or "")[:180]
    except Exception as e:
        judge_ok, reason = kw_ok, f"judge fallback ({type(e).__name__}): keywords used"

    grade_with_gemini_md.last = {"reason": reason}
    return judge_ok, refused

def _extract_variants_from_notebook_outputs() -> dict:
    nb_candidates = [Path("notebook.ipynb")]
    if "BASE_DIR" in globals():
        nb_candidates.append(Path(BASE_DIR) / "notebook.ipynb")
    if "ALT_ROOTS" in globals():
        for root in ALT_ROOTS:
            nb_candidates.append(Path(root) / "notebook.ipynb")
    nb_file = next((p for p in nb_candidates if p.exists() and p.is_file()), None)
    if nb_file is None:
        return {}
    try:
        nb_data = json.loads(nb_file.read_text(encoding="utf-8"))
    except Exception:
        return {}
    extracted = {}
    for cell in nb_data.get("cells", []):
        if cell.get("metadata", {}).get("id") != "Y0hWhO1u10Hm":
            continue
        cur_var = None
        for out in cell.get("outputs", []):
            if out.get("output_type") == "stream":
                txt = "".join(out.get("text", []))
                m = re.search(r"=====\s*Variant:\s*(\w+)\s*=====", txt)
                if m:
                    cur_var = m.group(1).strip()
            elif out.get("output_type") == "display_data" and cur_var:
                html_str = "".join(out.get("data", {}).get("text/html", []))
                tbody_m = re.search(r"<tbody>(.*?)</tbody>", html_str, flags=re.DOTALL)
                if not tbody_m:
                    continue
                rows_list = []
                for tr_m in re.finditer(r"<tr>(.*?)</tr>", tbody_m.group(1), flags=re.DOTALL):
                    tds = [
                        html.unescape(td_m.group(1))
                        for td_m in re.finditer(r"<td[^>]*>(.*?)</td>", tr_m.group(1), flags=re.DOTALL)
                    ]
                    if len(tds) == 5:
                        qid, q_txt, ans_txt, res_txt, reason_txt = tds
                        qid = qid.strip()
                        q_meta = QUESTIONS_GEMINI_MAP.get(qid, {})
                        rows_list.append({
                            "judge_reason": reason_txt.strip(),
                            "label": cur_var,
                            "run": 1,
                            "id": qid,
                            "tier": q_meta.get("tier", ""),
                            "question": q_txt.strip(),
                            "answer": ans_txt,
                            "correct": res_txt.strip() == "[OK]",
                            "refused": bool(REFUSAL_RE.search(norm(ans_txt))),
                        })
                if rows_list:
                    df_v = pd.DataFrame(rows_list)
                    out_p = Path(f"results/results_{cur_var}.csv")
                    out_p.parent.mkdir(parents=True, exist_ok=True)
                    df_v.to_csv(out_p, index=False)
                    extracted[cur_var] = out_p
                cur_var = None
    return extracted

def _find_variant_csv(label: str):
    candidates = [
        Path(f"results/results_{label}.csv"),
        Path(f"results/{label}_r1.csv"),
    ]
    if "BASE_DIR" in globals():
        candidates.append(Path(BASE_DIR) / "results" / f"results_{label}.csv")
        candidates.append(Path(BASE_DIR) / "results" / f"{label}_r1.csv")
    if "ALT_ROOTS" in globals():
        for root in ALT_ROOTS:
            candidates.append(Path(root) / "results" / f"results_{label}.csv")
            candidates.append(Path(root) / "results" / f"{label}_r1.csv")
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    extracted = _extract_variants_from_notebook_outputs()
    return extracted.get(label)

REUSE_EXISTING_RAG_ANSWERS = True

summary_rows_md = []
for label, fn in VARIANTS.items():
    existing_csv = _find_variant_csv(label) if REUSE_EXISTING_RAG_ANSWERS else None

    if existing_csv is not None:
        d = pd.read_csv(existing_csv)
        default_correct_cnt = int(d["correct"].sum()) if "correct" in d.columns else None
        new_correct, new_refused, new_reasons = [], [], []
        for _, row in d.iterrows():
            qid = str(row["id"])
            q_md = QUESTIONS_GEMINI_MAP.get(qid)
            if q_md is None:
                new_correct.append(bool(row.get("correct", False)))
                new_refused.append(bool(row.get("refused", False)))
                new_reasons.append(str(row.get("judge_reason", "")))
                continue
            ok, ref = grade_with_gemini_md(str(row["answer"]), q_md)
            new_correct.append(ok)
            new_refused.append(ref)
            new_reasons.append(grade_with_gemini_md.last.get("reason", ""))
        d["correct"] = new_correct
        d["refused"] = new_refused
        d["judge_reason"] = new_reasons
    elif "col" in globals():
        default_correct_cnt = None
        rows_md = []
        for qi, q_md in enumerate(QUESTIONS_GEMINI_MD):
            tr, seed = Trace(), SEED_BASE + 1000 + qi
            t0 = time.time()
            q_used, hits = fn(q_md, col, tr, seed)
            answer = generate_answer(q_used, hits, tr, seed)
            latency = time.time() - t0

            ok, ref = grade_with_gemini_md(answer, q_md)
            reason = grade_with_gemini_md.last.get("reason", "")

            docs = {h["doc"] for h in hits}
            recall = (sum(doc in docs for doc in q_md["gold_docs"]) / len(q_md["gold_docs"])) if q_md["gold_docs"] else np.nan
            ctx = norm(format_context(hits))
            ctx_keys = all(has_kw(ctx, kw) for kw in q_md["must_contain"]) if q_md["must_contain"] else np.nan

            rows_md.append({
                "judge_reason": reason,
                "label": label, "run": 1, "id": q_md["id"], "tier": q_md["tier"], "question": q_md["question"],
                "question_used": q_used, "queries": json.dumps(tr.queries, ensure_ascii=False),
                "retrieved": " | ".join(f"{h['id']}({h['distance']:.3f})" for h in hits),
                "gold_docs": ";".join(q_md["gold_docs"]), "gold_recall": recall, "ctx_has_keys": ctx_keys,
                "answer": answer, "correct": ok, "refused": ref,
                "llm_calls": tr.llm_calls, "embed_calls": tr.embed_calls, "latency_s": round(latency, 2),
            })
        d = pd.DataFrame(rows_md)
    else:
        print(f"[WARNING] No cached CSV found for variant '{label}' and vector collection 'col' is not initialized. Skipping.")
        continue

    out_csv_path = f"results/results_{label}_gemini_md.csv"
    d.to_csv(out_csv_path, index=False)
    if "BASE_DIR" in globals() and Path(BASE_DIR).resolve() != Path(".").resolve():
        (Path(BASE_DIR) / "results").mkdir(parents=True, exist_ok=True)
        d.to_csv(Path(BASE_DIR) / "results" / f"results_{label}_gemini_md.csv", index=False)

    t = d[["id", "question", "answer"]].copy()
    t["result"] = d["correct"].map({True: "[OK]", False: "[FAIL]"})
    t["judge_reason"] = d["judge_reason"]

    correct_cnt, total_cnt = int(d["correct"].sum()), len(d)
    summary_entry = {
        "variant": label,
        "correct": correct_cnt,
        "total": total_cnt,
        "accuracy": f"{correct_cnt / total_cnt:.0%}",
    }
    if default_correct_cnt is not None and total_cnt > 0:
        summary_entry["default_judge_accuracy"] = f"{default_correct_cnt / total_cnt:.0%}"
    summary_rows_md.append(summary_entry)

    print(f"\n===== Variant: {label} (Gemini Notebook MD Reference) =====")
    display(t.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    print(f"Accuracy: {correct_cnt}/{total_cnt} = {correct_cnt / total_cnt:.0%}")

if summary_rows_md:
    print("\n===== SUMMARY COMPARISON (GEMINI NOTEBOOK MD REFERENCE) =====")
    display(pd.DataFrame(summary_rows_md).style.hide(axis="index"))


===== Variant: baseline (Gemini Notebook MD Reference) =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides data in FP16 format, and this node needs to be handled specially during the conversion process. Specifically: - You should enclose the name of each node in double quotation marks ("""") and separate the nodes by semicolons (;). - For example, the command might look like this: `atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data""` This parameter is particularly useful when dealing with models that have specific input nodes providing FP16 data, ensuring that these nodes are correctly processed and that the input data format is maintained during the conversion to the `.om` model format.",[OK],The candidate answer aligns with the core technical facts provided in the Gemini Notebook answer.
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,"The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port, but without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port, and the port comes with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences reflect improvements and enhancements in the newer IT21VDMB version of the Atlas 200 DK, particularly in terms of power requirements and connectivity options.",[OK],Accurate and aligned with the core technical facts.
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. Here are the steps to address this issue: 1. **Check the Current Partition Space:** - Connect the SD card to the Ubuntu server over the USB port. - Run the `fdisk -l` command as the root user to check the current partition space. ```sh fdisk -l ``` - If the output shows: - Available sectors: 62,333,952 - Allocated sectors: 61,285,375 - Unallocated space: 512 MB (Calculated as: (62333952 - 61285375) x 512/1024/1024) - This indicates sufficient unallocated space. - Or, if the output shows: - Available sectors: 6,233,952 - Allocated sectors: 6,233,951 - This indicates insufficient unallocated space. 2. **Resize the Partition:** - If the unallocated space is insufficient, you need to resize the partition to create the required unallocated space. - Use a card reader to insert the SD card and connect it to the Ubuntu server over the USB port. - Alternatively, if a card reader is not available, place a jumper cap or wire over pins 16 and 18 on the Atlas 200 DK (power off the device first). - Run the following commands to install and start the GParted tool: ```sh apt-get install gparted gparted ``` - Switch to the corresponding device (SD card). - Unmount the partition: - Right-click /dev/sdb3 and choose Unmount. - Resize the partition: - Ensure the Unused space of the last partition is greater than 512 MB. - If the Unused space is smaller, delete unnecessary files from the /home/HwHiAiUser

Accuracy: 13/20 = 65%

===== Variant: stepback (Gemini Notebook MD Reference) =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],The answer aligns with the Gemini Notebook ground-truth and covers the key points accurately.
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],Accurate and aligned with the core technical facts.
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],The candidate answer aligns with the core technical facts provided in the Gemini Notebook answer.
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - N: Batch size - C1: Number of feature maps, which is derived from the original C dimension by dividing it by C0: \( C1 = \left\lceil \frac{C}{C0} \right\rceil \) - H: Height of the feature map - W: Width of the feature map - C0: The size of the matrix computing unit in the AI Core, which is equal to 16 for the FP16 type For the FP16 data type, C0 is 16. This means that the C dimension is divided into segments of 16 elements each, with the last segment possibly being padded with zeros if the division is not exact to ensure that the segments are of equal size.",[OK],"The answer correctly describes the structure of the NC1HWC0 format and the value of C0 for FP16, aligning with the Gemini Notebook answ

Accuracy: 13/20 = 65%

===== Variant: multiquery (Gemini Notebook MD Reference) =====


...

id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],The answer aligns with the Gemini Notebook ground-truth and covers the key points accurately.
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],Accurate and aligned with the core technical facts.
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],The candidate answer aligns with the core technical facts provided in the Gemini Notebook answer.
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - N: Batch size - C1: Number of feature maps, which is derived from the original C dimension by dividing it by C0: \( C1 = \left\lceil \frac{C}{C0} \right\rceil \) - H: Height of the feature map - W: Width of the feature map - C0: The size of the matrix computing unit in the AI Core, which is equal to 16 for the FP16 type For the FP16 data type, C0 is 16. This means that the C dimension is divided into segments of 16 elements each, with the last segment possibly being padded with zeros if the division is not exact to ensure that the segments are of equal size.",[OK],"The answer correctly describes the structure of the NC1HWC0 format and the value of C0 for FP16, aligning with the Gemini Notebook answ

Accuracy: 14/20 = 70%

===== Variant: combined (Gemini Notebook MD Reference) =====


id,question,answer,result,judge_reason
q1,What does the --input_fp16_nodes parameter do in the ATC tool?,"The `--input_fp16_nodes` parameter in the ATC tool is used to specify the name of an input node that has input data of type FP16 (half-precision floating point). This parameter is mutually exclusive with `--insert_op_conf`. When this parameter is set, it indicates that there is an input node in the original model that provides FP16 data. This is particularly useful when the input data to the model is expected to be in FP16 format, and the model needs to handle such inputs appropriately. Here are some key points about the `--input_fp16_nodes` parameter: - **Mutual Exclusivity**: It is mutually exclusive with `--insert_op_conf`, meaning you cannot use both parameters in the same ATC command. - **Usage Context**: It is typically used in scenarios where the input data to the model is expected to be in FP16 format, and the model needs to be configured to handle this input type. - **Example Usage**: You would specify the names of the input nodes that provide FP16 data, separated by semicolons. For example: ```sh atc --model=$HOME/test/resnet18.prototxt --weight=$HOME/test/resnet18.caffemodel --framework=0 --output=$HOME/test/out/caffe_resnet50 --input_fp16_nodes=""data1;data2"" ``` In summary, the `--input_fp16_nodes` parameter helps the ATC tool to correctly identify and handle input nodes that provide FP16 data, ensuring that the model conversion process accounts for this data type appropriately.",[OK],The answer aligns with the Gemini Notebook ground-truth and covers the key points accurately.
q2,What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?,The main hardware differences between the IT21DMDA (old PCB) and IT21VDMB (new PCB) versions of the Atlas 200 DK are summarized as follows: - **Power Input**: - IT21DMDA: Supports a power input range of 5 V to 28 V. - IT21VDMB: Uses a 12 V power input. - **MIPI-CSI Port**: - IT21DMDA: Equipped with a 22-pin MIPI-CSI port without cables. - IT21VDMB: Features a 51-pin MIPI-CSI port with cables. - **Number of GPIOs**: - IT21DMDA: Has 8 GPIOs. - IT21VDMB: Offers 11 GPIOs. These differences highlight the enhanced capabilities and potential applications supported by the newer IT21VDMB version compared to the IT21DMDA.,[OK],Accurate and aligned with the core technical facts.
q3,What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?,"The error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade typically indicates that there is insufficient unallocated space of 512 MB reserved on the SD card. This issue can be caused by using an old card creation script that did not allocate enough unallocated space. To resolve this, you need to ensure that the SD card has at least 512 MB of unallocated space. If the space is insufficient, you can use tools like GParted to modify the partition size and allocate the required space.",[OK],The candidate answer aligns with the core technical facts provided in the Gemini Notebook answer.
q4,How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?,"The NC1HWC0 5D tensor format is structured as follows: - **N**: Batch size - **C1**: Number of 16-element channels derived from the original C dimension - **H**: Height - **W**: Width - **C0**: Size of the matrix computing unit in AI Core, which is equal to 16 for the FP16 data type. This means that the C dimension is divided by 16, and any remainder is padded with zeros to align with the C0 size. In summary, for FP16 data type, C0 is 16, and C1 is calculated as \((C + 15) // 16\), ensuring that the data is stored consecutively in memory.",[OK],"The answer correctly describes the structure of the NC1HWC0 format and the value of C0 for FP16, aligning with the Gemini Notebook answer."
q5,Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8

Accuracy: 15/20 = 75%

===== SUMMARY COMPARISON (GEMINI NOTEBOOK MD REFERENCE) =====


variant,correct,total,accuracy,default_judge_accuracy
baseline,13,20,65%,80%
stepback,13,20,65%,70%
multiquery,15,20,75%,75%
memory,14,20,70%,75%
combined,15,20,75%,90%


# Playground Interativo do Sistema RAG

Esta seção permite realizar consultas interativas ao sistema RAG no Google Colab, testando qualquer uma das variantes de recuperação pré-consulta (`combined`, `baseline`, `stepback`, `multiquery`, `memory`). Por padrão, a variante combinada (`r_combined`) é utilizada por integrar resolução de histórico, abstração conceitual e fusão ponderada de rankings via RRF.

In [ ]:
# @title RAG Playground { run: "auto" }
# @markdown Digite sua pergunta técnica sobre o ecossistema Huawei Ascend e selecione a variante de recuperação:

question = "What does the --input_fp16_nodes parameter do in the ATC tool?"  # @param {type:"string"}
variant = "combined"  # @param ["combined", "baseline", "stepback", "multiquery", "memory"]
top_k = 4  # @param {type:"integer"}

def ask_rag_playground(
    question: str,
    variant: str = "combined",
    k: int = 4,
    history: list = None,
    col_instance = None,
    seed: int = 42,
) -> dict:
    """
    Interactive playground query runner for the Huawei Ascend RAG system.
    Supports variant selection ('combined', 'baseline', 'stepback', 'multiquery', 'memory').
    """
    if col_instance is None:
        if "col" not in globals() or col is None:
            raise RuntimeError(
                "ChromaDB collection 'col' is not initialized. "
                "Please run the indexing cells (Section 'Indexação Híbrida') first."
            )
        col_instance = col

    var_key = variant.strip().lower().replace("-", "").replace("_", "")
    key_mapping = {
        "combined": "combined",
        "rcombined": "combined",
        "stepback": "stepback",
        "rstepback": "stepback",
        "step_back": "stepback",
        "multiquery": "multiquery",
        "rmultiquery": "multiquery",
        "multi_query": "multiquery",
        "memory": "memory",
        "rmemory": "memory",
        "baseline": "baseline",
        "rbaseline": "baseline",
    }
    canonical_key = key_mapping.get(var_key, "combined")

    if canonical_key not in VARIANTS:
        available = ", ".join(list(VARIANTS.keys()))
        raise ValueError(f"Unknown variant '{variant}'. Available variants: {available}")

    fn = VARIANTS[canonical_key]
    tr = Trace()
    q_payload = {"question": question, "history": history or []}

    t0 = time.time()
    q_used, hits = fn(q_payload, col_instance, tr, seed)
    t_retrieval = time.time() - t0

    hits = hits[:k]

    t1 = time.time()
    answer = generate_answer(q_used, hits, tr, seed)
    t_gen = time.time() - t1
    t_total = time.time() - t0

    print("=" * 80)
    print(f"RAG PLAYGROUND - Variant: {canonical_key.upper()}")
    print("=" * 80)
    print(f"Input Question: {question}")
    if q_used != question:
        print(f"Effective Query (Rewritten): {q_used}")
    if tr.queries and len(tr.queries) > 1:
        print(f"Sub-Queries Generated ({len(tr.queries)}):")
        for sq in tr.queries:
            print(f"  - {sq}")
    print("-" * 80)
    print("Generated Answer:\n")
    print(answer)
    print("-" * 80)
    print(
        f"Latencies: Retrieval = {t_retrieval:.3f}s | Generation = {t_gen:.3f}s | Total = {t_total:.3f}s"
    )
    print(f"Calls: LLM calls = {tr.llm_calls} | Embedding calls = {tr.embed_calls}")
    print("\nRetrieved Context Chunks:")
    for rank, h in enumerate(hits, 1):
        tbl_flag = " [TABLE]" if h.get("is_table") else ""
        dist_info = f"dist={h.get('distance', 0.0):.4f}" if "distance" in h else f"rrf={h.get('rrf', 0.0):.4f}"
        print(f"  [{rank}] {h['doc']} | {h['section']} | {dist_info}{tbl_flag}")
    print("=" * 80)

    return {
        "question": question,
        "effective_query": q_used,
        "variant": canonical_key,
        "answer": answer,
        "hits": hits,
        "trace": tr,
        "latency_retrieval_s": round(t_retrieval, 3),
        "latency_gen_s": round(t_gen, 3),
        "latency_total_s": round(t_total, 3),
    }

playground_result = ask_rag_playground(
    question=question,
    variant=variant,
    k=top_k,
)
